# Figure 4 and Supplementary Figures S10–S13: APA and miRNA coordination

This notebook contains all analysis and plotting code for the individual panels and assembled figures. It starts from TMM-normalized gene logCPM and the authoritative gene regression tables used by Figures 2/3, per-sample DaPars prediction tables, processed miRNA quantifications, annotated target predictions, gene/GO resources, and fixed CHIRAL sample phases. It does not execute or import legacy notebooks or project scripts. CHIRAL phases represent inferred ordering of unordered samples; phase estimation is retained as a processed input.

**Rhythmicity criteria:** APA events and miRNAs use nominal **P < 0.05**. mRNA uses the exact versionless ENSG lists from Figures 2/3: **BH q < 0.05 and stored peak-to-trough `Amplitude > 0.1`**, based on TMM-normalized log2 CPM. The additional amplitude gate is retained to match those lists exactly; membership is checked against the saved significant-gene tables. Correlation and enrichment use **BH q < 0.05** within their stated test families. Phase alignment uses ±3 h around 0 h (in-phase) or 12 h (anti-phase). Input QC filters differ by assay and are printed below.

**Run environment:** Python 3 in the existing Docker Jupyter environment. Required packages: numpy, pandas, scipy, statsmodels, matplotlib, and IPython. Direct S10D coverage extraction additionally needs pysam and indexed BAMs. Set `CHRONO_PROJECT_ROOT` and/or `CHRONO_DATA_DIR` if data are stored separately. Input data are not embedded in this public notebook.

**S10D is enabled.** Configure `TMEM183A_BAM_DIR`, `BAM_PATHS`, and `TMEM183A_GTF` in the first code cell. The complete S10 includes three heatmaps and real TMEM183A coverage from four TCGA BAMs. If the original BAMs are unavailable, `TMEM183A_USE_SOURCE_CACHE=True` permits regeneration from checksum-validated per-base depth tables derived from those BAMs; no legacy image or simulated coverage is used.

All panels use globally configured text of at least 7 pt and 52.5 mm layout modules. Every generated individual panel and assembly is displayed inline. PNGs omit panel letters; PDFs include them. Output tables retain the observations and statistics needed to regenerate the figures.

**Statistical notation:** figure annotations, axes and color bars use italic lowercase *p* for raw p-values and *q* for FDR-adjusted values. Existing significance stars keep their original tests and thresholds.

In [ ]:
# Cell 1: Setup, portable paths, thresholds, and global figure style
from pathlib import Path
from collections import OrderedDict
import os, re, json, sys, platform, hashlib, warnings, importlib.metadata, time
import numpy as np
import pandas as pd
from scipy import stats, special
from statsmodels.stats.multitest import multipletests
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.text import Text
from IPython.display import display, Image, Markdown

SEED = 4404
P_THRESHOLD = 0.05  # Nominal APA and miRNA rhythmicity.
GENE_Q_THRESHOLD = 0.05
GENE_AMPLITUDE_MIN = 0.1  # Stored peak-to-trough amplitude, exactly as Figures 2/3.
Q_THRESHOLD = 0.05  # Adjusted correlation/enrichment/comparison tests.
GO_P_THRESHOLD = 0.01  # Panel H uses raw GO enrichment P.
GO_MAX_TERMS_PER_COHORT = 20
GO_DISPLAY_SCORE_MAX = 22.0  # Legacy visual cap; exported P values remain uncapped.
MIN_SAMPLES = 10
PHASE_WINDOW_H = 3.0
CORRELATION_PAIR_LIMIT = 20000
MODULE_MM = 52.5
DPI = 300
FONT = dict(base=8, tick=7, label=8, title=9, legend=7, annotation=7, panel=10)
FONT_FAMILY = 'DejaVu Sans'
assert min(FONT.values()) >= 7
np.random.seed(SEED)
mpl.rcParams.update({
    'font.family': FONT_FAMILY, 'font.size': FONT['base'],
    'axes.titlesize': FONT['title'], 'axes.labelsize': FONT['label'],
    'xtick.labelsize': FONT['tick'], 'ytick.labelsize': FONT['tick'],
    'legend.fontsize': FONT['legend'], 'legend.title_fontsize': FONT['legend'],
    'figure.titlesize': FONT['title'], 'axes.spines.top': False,
    'axes.spines.right': False, 'legend.frameon': False,
    'pdf.fonttype': 42, 'ps.fonttype': 42, 'savefig.dpi': DPI,
    'figure.dpi': 110, 'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.linewidth': 0.6, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6,
    'grid.linewidth': 0.4, 'grid.alpha': 0.35,
})

# Set either environment variable when publishing data separately from this repository.
_candidates = [Path(os.environ['CHRONO_PROJECT_ROOT'])] if os.environ.get('CHRONO_PROJECT_ROOT') else [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in _candidates if (p / 'data' / 'Merged').exists()), None)
if ROOT is None:
    ROOT = Path(os.environ.get('CHRONO_PROJECT_ROOT', Path.cwd())).resolve()
DATA_DIR = Path(os.environ.get('CHRONO_DATA_DIR', ROOT / 'data')).resolve()
if not DATA_DIR.exists():
    raise FileNotFoundError('Set CHRONO_PROJECT_ROOT or CHRONO_DATA_DIR to the supplied processed-data directory.')
ANALYSIS_SLUG = 'fig04_apa_mirna_coordination'
TABLE_DIR = ROOT / 'data' / 'publication' / ANALYSIS_SLUG
FIG_DIR = ROOT / 'data' / 'figures' / 'publication' / ANALYSIS_SLUG
LOG_DIR = ROOT / 'data' / 'logs' / 'publication' / ANALYSIS_SLUG
for p in [TABLE_DIR, FIG_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)
# S10D: edit this directory or the per-sample paths before extracting coverage.
# GRC/jptsvr default: the project-mounted data/figureS10D_bams directory.
TMEM183A_BAM_DIR = Path(os.environ.get('CHRONO_TMEM183A_BAM_DIR', DATA_DIR / 'figureS10D_bams'))
BAM_PATHS = {
    'TCGA-2J-AAB6-01A': TMEM183A_BAM_DIR / 'TCGA-2J-AAB6-01A.bam',
    'TCGA-YB-A89D-01A': TMEM183A_BAM_DIR / 'TCGA-YB-A89D-01A.bam',
    'TCGA-IB-AAUW-01A': TMEM183A_BAM_DIR / 'TCGA-IB-AAUW-01A.bam',
    'TCGA-HZ-7926-01A': TMEM183A_BAM_DIR / 'TCGA-HZ-7926-01A.bam',
}
TMEM183A_GTF = Path(os.environ.get('CHRONO_TMEM183A_GTF', DATA_DIR / 'Ref/gencode.v49.basic.annotation.gtf'))
# Re-extract whenever all BAMs are present. When none are present, True permits
# regeneration from the checksum-validated, real BAM-derived source tables.
# Set False to require original BAMs; partial/malformed BAM sets always fail.
TMEM183A_USE_SOURCE_CACHE = True

COHORTS = OrderedDict([
    ('TCGA', dict(label='TCGA-PAAD tumor', short='TCGA', color='#D55E00', mirna_slug='tcga_paad_tumor')),
    ('CPTAC', dict(label='CPTAC-3 tumor', short='CPTAC', color='#009E73', mirna_slug='cptac3_tumor')),
    ('GTEx', dict(label='GTEx normal', short='GTEx', color='#0072B2', mirna_slug='gtex_normal')),
])
GROUP_COLORS = {'Tumor only':'#D55E00', 'Mutual/shared':'#009E73', 'Normal only':'#0072B2'}
GROUP_COLORS.update({'tumor-only':'#D55E00','tumor_only':'#D55E00','shared':'#009E73','normal-only':'#0072B2','normal_only':'#0072B2'})
INPUT_RECORDS = {}
def record_input(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Required input is missing: {path}')
    try:
        name = str(path.relative_to(DATA_DIR))
    except ValueError:
        name = path.name
    INPUT_RECORDS[name] = {'relative_path': name, 'bytes': path.stat().st_size, 'kind': 'file' if path.is_file() else 'directory'}
    return path

def save_table(df, name):
    path = TABLE_DIR / (name if str(name).endswith('.tsv') else f'{name}.tsv')
    df.to_csv(path, sep='\t', index=False)
    return path

def significant(df):
    if 'is_rhythmic' not in df:
        raise KeyError('Set the modality-specific is_rhythmic flag before selection.')
    return df.loc[df['is_rhythmic'].fillna(False).astype(bool) & np.isfinite(df['phase'])].copy()

def circular_lag(a, b):
    return (np.asarray(a) - np.asarray(b) + 12.) % 24. - 12.

def circular_density(phases, bandwidth=0.3):
    values = np.asarray(phases, dtype=float)
    values = values[np.isfinite(values)] % 24
    theta = np.linspace(0, 2*np.pi, 361)
    if not len(values):
        return theta, np.zeros_like(theta)
    kappa = 1.0 / bandwidth**2
    density = np.zeros(len(theta))
    for chunk in np.array_split(values, max(1, int(np.ceil(len(values)/2000)))):
        density += np.exp(kappa * (np.cos(theta[:,None] - chunk[None,:]*np.pi/12.) - 1)).sum(axis=1)
    density /= len(values) * 2*np.pi * special.i0e(kappa)
    return theta, density

def phase_axis(phases):
    values = np.asarray(phases, float)
    values = values[np.isfinite(values)]
    if not len(values):
        return np.nan, np.nan
    z = np.mean(np.exp(2j * values * np.pi/12))
    return (np.angle(z) % (2*np.pi)) * 6/np.pi, abs(z)

def style_polar(ax):
    ax.set_theta_zero_location('N'); ax.set_theta_direction(-1)
    ax.set_xticks(np.arange(0,24,4)*np.pi/12)
    ax.set_xticklabels([str(x) for x in range(0,24,4)])
    ax.set_facecolor('#F3F4F7'); ax.grid(True, alpha=.5)

thresholds = pd.DataFrame([
    {'component':'APA / miRNA rhythmicity','criterion':'Nominal P < 0.05; no additional harmonic amplitude gate'},
    {'component':'mRNA rhythmicity','criterion':'Same ENSG sets as Figures 2/3: BH q < 0.05 AND stored Amplitude > 0.1; TMM log2 CPM'},
    {'component':'APA / miRNA harmonic model','criterion':'24 h intercept + cosine + sine; finite n >= 10; F survival function'},
    {'component':'Gene expression','criterion':'Use processed TMM log2 CPM and authoritative all-gene regression tables; verify membership against previous significant tables'},
    {'component':'Correlation / other enrichment significance','criterion':'BH q < 0.05 within each stated test family'},
    {'component':'Figure 4B/C display','criterion':'Raw Mann–Whitney P (B) and one-sided Fisher overlap P (C); BH q retained in source tables'},
    {'component':'Figure 4D/I comparisons','criterion':'Raw two-sided Fisher P; annotate P < 0.05 only'},
    {'component':'Figure 4H GO display','criterion':f'Raw P < {GO_P_THRESHOLD}; at most {GO_MAX_TERMS_PER_COHORT} lowest-P terms per cohort'},
    {'component':'Phase alignment','criterion':'in-phase within 3 h of 0; anti-phase within 3 h of 12 h'},
    {'component':'APA input QC','criterion':'legacy >=50% prevalence, long/short >=20 when available, PDUI SD>=0.05 and range>=0.10'},
    {'component':'miRNA input QC','criterion':'>=60% prevalence at >=1 CPM/TPM; GDC additionally >=50 reads and no cross-mapping; log2 SD>=0.15, range>=1'},
    {'component':'Correlation population','criterion':'Top 20,000 eligible unique miRNA-key/ENSG pairs, ranked by -log10(miRNA P)-log10(gene P)'},
])
save_table(thresholds, 'analysis_thresholds')
display(thresholds)
print('APA/miRNA: nominal P <',P_THRESHOLD,'| genes: BH q <',GENE_Q_THRESHOLD,'AND stored Amplitude >',GENE_AMPLITUDE_MIN)


## Processed inputs, phase alignment, and shared harmonic regression

Gene membership and acrophases come from the processed all-gene regression tables used by Figures 2/3. Versionless ENSG identifiers define the membership and all host-gene/target joins. Curated symbol panels select one representative per symbol deterministically (lowest q, then ENSG). The original gene lists additionally require stored peak-to-trough `Amplitude > 0.1`, beyond q < 0.05. No gene selection is recomputed from TPM.

Gene expression is already TMM-normalized log2 CPM. The original non-GEO TMM matrix and non-GEO CHIRAL phases numerically reproduce the stored gene regression results (peak-to-trough amplitude and acrophase); all three modalities therefore use the same verified phase input. APA and miRNA are fitted using intercept + cosine + sine with period 24 h; at least ten finite observations and a nonconstant row are required. Both raw P and BH q are saved, but their rhythmicity selection uses raw P. Sample aliases are canonicalized, and sample matching is exported in QC tables. A TMM harmonic fit supplies display coefficients for the CXCL12 curve; stored gene membership, significance and acrophase remain authoritative, and fit agreement is exported. Core clock genes are printed as a diagnostic, not another selection gate.

In [ ]:
# Cell 2: Load authoritative TMM gene lists, verified sample phases, and clock controls
def harmonic_fit(matrix, phases=None):
    """Vectorized 24 h harmonic regression; defaults to the APA/miRNA phase map."""
    phases = PHASE if phases is None else phases
    columns = [str(c) for c in matrix.columns if str(c) in phases.index]
    if len(columns) < MIN_SAMPLES:
        raise ValueError(f'Only {len(columns)} phase-matched samples; at least {MIN_SAMPLES} required.')
    matrix = matrix.loc[:, columns]
    phi = phases.loc[columns].to_numpy(float) * np.pi / 12
    design = np.column_stack([np.ones(len(phi)), np.cos(phi), np.sin(phi)])
    records = []
    for start in range(0, len(matrix), 2048):
        y = matrix.iloc[start:start+2048].to_numpy(dtype=float)
        mask = np.isfinite(y)
        n = mask.sum(axis=1)
        yy = np.where(mask, y, 0.)
        sum_y = yy.sum(axis=1)
        sst = np.sum(yy*yy, axis=1) - sum_y**2 / np.maximum(n,1)
        xtx = np.einsum('ni,mn,nj->mij', design, mask.astype(float), design, optimize=True)
        xty = yy @ design
        beta = np.einsum('mij,mj->mi', np.linalg.pinv(xtx), xty)
        fitted = beta @ design.T
        sse = np.sum(np.where(mask, (yy-fitted)**2, 0.), axis=1)
        ssr = np.maximum(sst-sse,0.)
        valid = (n >= MIN_SAMPLES) & (sst > 1e-12) & (np.linalg.matrix_rank(xtx) == 3)
        f = np.divide(ssr/2, sse/np.maximum(n-3,1), out=np.full(len(n), np.inf), where=sse>0)
        p = stats.f.sf(f, 2, np.maximum(n-3,1))
        amplitude = np.hypot(beta[:,1], beta[:,2])
        sd = np.sqrt(np.maximum(sst,0)/np.maximum(n-1,1))
        out = pd.DataFrame({'n':n, 'beta0':beta[:,0], 'beta_cos':beta[:,1], 'beta_sin':beta[:,2],
                            'amplitude':amplitude, 'relative_amplitude':amplitude/np.maximum(sd,1e-300),
                            'phase':np.mod(np.arctan2(beta[:,2],beta[:,1]),2*np.pi)*12/np.pi, 'p':p},
                           index=matrix.index[start:start+len(y)])
        records.append(out.loc[valid])
    result = pd.concat(records) if records else pd.DataFrame()
    if result.empty:
        raise ValueError('No variable features with enough valid observations to fit.')
    result['q'] = multipletests(result['p'], method='fdr_bh')[1]
    result.index.name = 'feature'
    return result

def canonical_sample(value):
    value=str(value).strip().replace('"','')
    candidates=[v.strip() for v in value.split(',')]
    mapped={SAMPLE_CANONICAL[v] for v in candidates if v in SAMPLE_CANONICAL}
    if len(mapped)>1:
        raise ValueError(f'Compound sample maps to conflicting phase groups: {value}')
    return next(iter(mapped)) if mapped else value

def _strip_ensg(values):
    return pd.Series(values, copy=False).astype(str).str.replace(r'\..*$', '', regex=True)


def _sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def _load_authoritative_regression(group):
    all_path = record_input(DATA_DIR / 'Merged' / 'regression' / f'all_genes_regression_periodic_12h_{group}.txt')
    sig_path = record_input(DATA_DIR / 'Merged' / 'regression' / f'sig_genes_regression_periodic_12h_{group}.txt')
    required = {'Gene', 'ENSG', 'Amplitude', 'p', 'q', 'acrophase'}
    all_raw, sig_raw = pd.read_csv(all_path, sep='\t'), pd.read_csv(sig_path, sep='\t')
    if not required.issubset(all_raw.columns) or not required.issubset(sig_raw.columns):
        raise ValueError(f'{group}: regression table misses required columns.')
    def clean(frame):
        out = pd.DataFrame({
            'gene': frame.Gene.astype(str).str.strip().str.upper(),
            'ensg': _strip_ensg(frame.ENSG),
            'amplitude': pd.to_numeric(frame.Amplitude, errors='coerce'),
            'p': pd.to_numeric(frame.p, errors='coerce'),
            'q': pd.to_numeric(frame.q, errors='coerce'),
            'phase': pd.to_numeric(frame.acrophase, errors='coerce') % 24,
        })
        if out.ensg.duplicated().any():
            raise ValueError(f'{group}: duplicate versionless ENSG identifiers.')
        out['is_rhythmic'] = (out.q < GENE_Q_THRESHOLD) & (out.amplitude > GENE_AMPLITUDE_MIN)
        return out.set_index('ensg', drop=False)
    all_rows, sig_rows = clean(all_raw), clean(sig_raw)
    if set(all_rows.index[all_rows.is_rhythmic]) != set(sig_rows.index):
        raise ValueError(f'{group}: saved sig-gene ENSG set mismatches q/amplitude selection.')
    return all_rows, all_path, sig_path


def load_authoritative_gene_data():
    """Replace mRNA refitting with the Figure 2/3 TMM/regression contract.

    ``PHASE`` is the non-GEO phase map used by APA, miRNA, and the source TMM fits.
    ``GENE_PHASE`` is an alias retained for display-curve call sites.
    """
    global T2G, EXPR, TPM, GENE, GENE_ALL, EXPR_ENSG, GENE_PHASE, GENE_DISPLAY_FITS

    t2g_path = record_input(DATA_DIR / 'Ref' / 't2g.txt')
    t2g = pd.read_csv(t2g_path, sep='\t', usecols=['TXNAME', 'GENEID', 'GENENAME'], dtype=str)
    T2G = pd.DataFrame({'transcript': t2g.TXNAME.str.split('|').str[0].str.split('.').str[0],
                        'ensg': t2g.GENEID.str.split('.').str[0],
                        'gene': t2g.GENENAME.str.upper()}).dropna().drop_duplicates()

    meta_path = record_input(DATA_DIR / 'Merged' / 'GTEx_TCGA_merged_meta.csv')
    # This is the original 25,633-gene TMM matrix used by the saved regression tables.
    # The later GEO-labelled matrix has 24,978 rows and omits CXCL12.
    tmm_path = record_input(DATA_DIR / 'Merged' / 'GTEx_TCGA_merged_TMM_unfiltered_expression.csv')
    meta = pd.read_csv(meta_path, sep='\t', index_col=0)
    meta.index.name = 'Sample'
    meta = meta.reset_index()
    if not {'Sample', 'Info'}.issubset(meta.columns) or meta.Sample.duplicated().any():
        raise ValueError('Merged metadata must contain unique Sample and Info columns.')
    group_map = {'TCGA': 'TCGA-PAAD_Tumor', 'CPTAC': 'CPTAC-3_Tumor', 'GTEx': 'GTEx_Normal'}
    meta = meta[meta.Info.isin(group_map.values())].copy()
    expression_rows = []
    for row in meta.itertuples(index=False):
        sample = str(row.Sample).replace('"', '').strip()
        canonical = canonical_sample(sample)
        phase = PHASE.get(canonical, np.nan)
        if not np.isfinite(phase):
            raise ValueError(f'Cannot resolve non-GEO CHIRAL phase for metadata sample {sample}.')
        expression_rows.append({'input_sample_id': sample, 'sample_id': canonical, 'Group': row.Info,
                                'phase_h': float(phase), 'phase_source': 'CHIRAL_dual_penalty_phi_12h.csv'})
    expression_samples = pd.DataFrame(expression_rows)
    if expression_samples.sample_id.duplicated().any():
        bad = expression_samples.loc[expression_samples.sample_id.duplicated(False), ['input_sample_id', 'sample_id']]
        raise ValueError(f'Canonical mRNA sample IDs collide: {bad.head().to_dict("records")}')
    GENE_PHASE = PHASE
    save_table(expression_samples, 'gene_expression_sample_phase_mapping')
    save_table(expression_samples[['input_sample_id', 'sample_id', 'Group', 'phase_h']],
               'gene_expression_phase_concordance')

    header = pd.read_csv(tmm_path, nrows=0)
    id_column = header.columns[0]
    input_samples = expression_samples.input_sample_id.tolist()
    missing = sorted(set(input_samples) - set(header.columns))
    if missing:
        raise ValueError(f'TMM matrix lacks phase-matched samples: {missing[:5]}')
    tmm = pd.read_csv(tmm_path, usecols=[id_column, *input_samples], index_col=id_column)
    tmm.index = _strip_ensg(tmm.index).to_numpy()
    if tmm.index.duplicated().any():
        raise ValueError('TMM matrix has duplicate versionless ENSG identifiers.')
    tmm = tmm.apply(pd.to_numeric, errors='coerce')
    tmm.columns = expression_samples.set_index('input_sample_id').loc[tmm.columns, 'sample_id'].tolist()
    if tmm.columns.duplicated().any():
        raise ValueError('TMM matrix columns collide after canonical sample mapping.')

    EXPR_ENSG, EXPR, TPM, GENE_ALL, GENE, GENE_DISPLAY_FITS = {}, {}, {}, {}, {}, {}
    inputs = [t2g_path, meta_path, tmm_path]
    qc, duplicate_audit = [], []
    for cohort, group in group_map.items():
        all_rows, all_path, sig_path = _load_authoritative_regression(group)
        inputs.extend([all_path, sig_path])
        samples = expression_samples.loc[expression_samples.Group.eq(group), 'sample_id'].tolist()
        expression = tmm.loc[:, samples].copy()
        all_rows['n'] = expression.reindex(all_rows.index).notna().sum(axis=1).to_numpy(int)
        EXPR_ENSG[cohort], GENE_ALL[cohort] = expression, all_rows

        # Symbol panels use one reproducible representative; all memberships remain ENSG based.
        ranked = all_rows.reset_index(drop=True).sort_values(['gene', 'q', 'ensg'], kind='mergesort')
        duplicate = ranked[ranked.gene.duplicated(False)].copy()
        if len(duplicate):
            duplicate['cohort'] = cohort
            duplicate['representative_rule'] = 'lowest q, then ENSG lexical order'
            duplicate_audit.append(duplicate)
        symbols = ranked.drop_duplicates('gene', keep='first').set_index('gene', drop=False)
        GENE[cohort] = symbols
        EXPR[cohort] = expression.reindex(symbols.ensg).set_axis(symbols.index, axis=0)
        TPM[cohort] = None  # compatibility name only: all mRNA values are TMM logCPM.

        display_ensg = harmonic_fit(expression, phases=GENE_PHASE).rename_axis('ensg')
        display_symbol = display_ensg.reindex(symbols.ensg).set_axis(symbols.index, axis=0)
        GENE_DISPLAY_FITS[cohort] = display_symbol
        verification = all_rows.join(display_ensg.add_prefix('display_'), how='left')
        verification['display_phase_minus_stored_h'] = circular_lag(verification.display_phase, verification.phase)
        verification['display_peak_to_trough_minus_stored'] = 2 * verification.display_amplitude - verification.amplitude
        comparable = verification.loc[verification.is_rhythmic & verification.display_amplitude.notna()]
        if (comparable.display_peak_to_trough_minus_stored.abs() > 1e-7).any() or \
           (comparable.display_phase_minus_stored_h.abs() > 1e-7).any():
            raise ValueError(f'{group}: source TMM/phase fit does not reproduce authoritative significant-gene metrics.')
        save_table(verification.reset_index(drop=True), f'{cohort}_gene_display_fit_verification')
        save_table(all_rows.reset_index(drop=True), f'{cohort}_authoritative_gene_membership')
        save_table(symbols.reset_index(drop=True), f'{cohort}_gene_symbol_representatives')
        qc.append({'cohort': cohort, 'group': group, 'tmm_genes': expression.shape[0], 'tmm_samples': expression.shape[1],
                   'regression_tested_genes': len(all_rows), 'rhythmic_q05_amplitude_gt_01': int(all_rows.is_rhythmic.sum()),
                   'symbol_representatives': len(symbols), 'duplicate_symbol_rows': len(duplicate),
                   'display_phase_abs_median_delta_h': float(np.nanmedian(np.abs(verification.display_phase_minus_stored_h))),
                   'display_peak_to_trough_abs_median_delta': float(np.nanmedian(np.abs(verification.display_peak_to_trough_minus_stored)))})
        print(cohort, 'TMM logCPM:', expression.shape, '| authoritative rhythmic:', int(all_rows.is_rhythmic.sum()), flush=True)
    save_table(pd.DataFrame(qc), 'gene_input_qc')
    save_table(pd.concat(duplicate_audit, ignore_index=True) if duplicate_audit else pd.DataFrame(), 'gene_symbol_duplicate_audit')
    save_table(pd.DataFrame([{'relative_path': str(Path(p).relative_to(DATA_DIR)), 'bytes': Path(p).stat().st_size,
                              'sha256': _sha256(p)} for p in dict.fromkeys(inputs)]), 'authoritative_gene_input_manifest')

    clock_genes = ['ARNTL', 'CLOCK', 'NPAS2', 'CRY1', 'CRY2', 'PER1', 'PER2', 'PER3', 'NR1D1', 'NR1D2', 'DBP', 'TEF']
    clock = []
    for cohort, frame in GENE.items():
        display_ids = [('BMAL1' if gene == 'ARNTL' and 'BMAL1' in frame.index else gene) for gene in clock_genes]
        z = frame.reindex(display_ids).copy()
        z['clock_label'] = clock_genes
        z['cohort'], z['present'] = cohort, z.ensg.notna()
        clock.append(z.reset_index(drop=True))
    clock = pd.concat(clock, ignore_index=True)
    save_table(clock, 'core_clock_gene_controls')
    display(pd.DataFrame(qc))
    display(clock[['cohort', 'gene', 'ensg', 'n', 'phase', 'amplitude', 'q', 'is_rhythmic', 'present']])


def load_non_geo_phase_data():
    """Initialize only the historical non-GEO aliases required by APA and miRNA."""
    global PHASE, SAMPLE_CANONICAL
    phase_path = record_input(DATA_DIR / 'Merged' / 'CHIRAL_dual_penalty_phi_12h.csv')
    raw = pd.read_csv(phase_path)
    rows = []
    SAMPLE_CANONICAL = {}
    for row in raw.itertuples(index=False):
        source = str(row.sample).replace('"', '').strip()
        canonical = source.split(',')[0].strip()
        for alias in source.split(','):
            alias = alias.strip()
            SAMPLE_CANONICAL[alias] = canonical
            rows.append({'sample': alias, 'phase': float(row.phi) % (2 * np.pi) * 12 / np.pi})
    phases = pd.DataFrame(rows).dropna()
    if (phases.groupby('sample').phase.agg(lambda x: np.ptp(x)) > 1e-8).any():
        raise ValueError('Non-GEO CHIRAL aliases have conflicting phase values.')
    PHASE = phases.drop_duplicates('sample').set_index('sample').phase
    save_table(PHASE.rename_axis('sample').reset_index(), 'sample_phases')
    save_table(pd.DataFrame(SAMPLE_CANONICAL.items(), columns=['sample_alias', 'canonical_sample']), 'sample_alias_mapping')


def load_base_data():
    """Initialize the separated APA/miRNA and authoritative mRNA input contracts."""
    load_non_geo_phase_data()
    load_authoritative_gene_data()


load_base_data()


In [ ]:
# Cell 3: Shared individual-panel and assembly export functions
PANELS = OrderedDict()
EXPORT_RECORDS = []
def register_panel(figid, letter, content, draw, width=1, height=1):
    if width not in (1,2,3,4) or height < 1 or int(height) != height:
        raise ValueError('Panel sizes must use whole 52.5 mm modules, with widths 1–4.')
    content=re.sub(r'[^a-z0-9]+', '_', str(content).lower()).strip('_')
    key=(str(figid),str(letter))
    if key in PANELS:
        raise ValueError(f'Duplicate panel: {key}')
    PANELS[key]=dict(figid=str(figid), letter=str(letter), content=content, draw=draw, width=int(width), height=int(height))

FIGURE_CONTENT={'4':'apa_mirna_coordination','S10':'apa_landscape','S11':'apa_machinery','S12':'apa_expression_phase_lags','S13':'mirna_target_relationships'}

def _new_canvas(width,height):
    fig=plt.figure(figsize=(width*MODULE_MM/25.4,height*MODULE_MM/25.4),layout='constrained')
    fig.get_layout_engine().set(w_pad=0.035,h_pad=0.035,wspace=0.02,hspace=0.02)
    return fig

def _export_canvas(fig,stem,figid,panels,width,height,canvases,assembled=False):
    # Fixed-position polar panels already reserve their title/key space.
    if not assembled and all(ax.get_subplotspec() is None for ax in fig.axes):
        fig.set_layout_engine(None)
    fig.canvas.draw()
    text_sizes=[t.get_fontsize() for t in fig.findobj(Text) if t.get_visible() and t.get_text().strip()]
    if text_sizes and min(text_sizes) < 7-1e-6:
        raise AssertionError(f'{stem}: text smaller than 7 pt: {min(text_sizes)}')
    folder=FIG_DIR / ('figure4' if figid=='4' else f'supplementary{figid[1:]}')
    folder.mkdir(parents=True,exist_ok=True)
    png=folder / f'{stem}.png'; pdf=folder / f'{stem}.pdf'
    fig.savefig(png,dpi=DPI,facecolor='white')
    tags=[]
    for canvas,panel in zip(canvases,panels):
        tags.append(canvas.text(0.008,0.995,panel['letter'],ha='left',va='top',fontweight='bold',fontsize=FONT['panel']))
    fig.savefig(pdf,facecolor='white')
    for tag in tags:
        tag.remove()
    complete=(figid!='S10' or ('S10','D') in PANELS)
    if assembled:
        EXPORT_RECORDS[:]=[r for r in EXPORT_RECORDS if not (r['figure']==figid and r['panel']=='assembled')]
    EXPORT_RECORDS.append({'stem':stem,'figure':figid,'panel':'assembled' if assembled else panels[0]['letter'],
        'width_mm':width*MODULE_MM,'height_mm':height*MODULE_MM,'minimum_font_pt':min(text_sizes) if text_sizes else np.nan,
        'png_has_panel_letters':False,'pdf_has_panel_letters':True,'assembly_complete':complete if assembled else True,
        'omitted_panel':'S10D (BAM paths required)' if assembled and not complete else '',
        'png':str(png.relative_to(ROOT)),'pdf':str(pdf.relative_to(ROOT))})
    display(Image(filename=str(png),width=min(1000,int(width*240))))
    plt.close(fig)

def _pack_rows(panels):
    placements=[]; x=0; y=0; row_height=0
    for panel in panels:
        if x+panel['width']>4:
            y+=row_height; x=0; row_height=0
        placements.append((panel,x,y))
        x+=panel['width']; row_height=max(row_height,panel['height'])
    return placements,y+row_height

def export_figure(figid):
    """Export all registered components and an assembly without rescaling panel text."""
    figid=str(figid)
    panels=sorted([p for (f,l),p in PANELS.items() if f==figid],key=lambda p:p['letter'])
    if not panels:
        raise ValueError(f'No panels registered for {figid}')
    display(Markdown(f'### Figure {figid}'))
    if figid=='S10' and ('S10','D') not in PANELS:
        display(Markdown('**Incomplete assembly:** the heatmaps below are generated; the RNA-seq coverage panel is disabled until BAM paths are supplied. The optional coverage cell contains all required code.'))
    for panel in panels:
        display(Markdown(f"**Panel {panel['letter']} — {panel['content'].replace('_',' ')}**"))
        fig=_new_canvas(panel['width'],panel['height'])
        panel['draw'](fig)
        stem=f"Figure{figid}{panel['letter']}_{panel['content']}"
        _export_canvas(fig,stem,figid,[panel],panel['width'],panel['height'],[fig])
    placements,height=_pack_rows(panels)
    width=max(x+panel["width"] for panel,x,y in placements)
    fig=_new_canvas(width,height)
    grid=fig.add_gridspec(height,24 if figid=='S10' else width,wspace=0.01,hspace=0.01)
    canvases=[]
    s10_heatmap_image=None
    # One shared machinery key across E/F leaves both clock faces larger.
    ef_canvases={}
    if figid=='4':
        ef_block=fig.add_subfigure(grid[1:2,0:2])
        ef_pair=ef_block.subfigures(1,2,wspace=0)
        ef_canvases=dict(zip('EF',ef_pair))
        _machinery_legend(ef_block,columns=4)
    for panel,x,y in placements:
        if panel['letter'] in ef_canvases:
            canvas=ef_canvases[panel['letter']]
            _draw_machinery_polar(canvas,'GTEx' if panel['letter']=='E' else 'TCGA',compact=True,show_legend=False)
        else:
            if figid=='S10':
                # Contiguous one-module heatmaps; one shared scale at the right.
                is_heatmap=panel['letter'] in 'ABC'
                left=2+6*'ABC'.index(panel['letter']) if is_heatmap else 0
                right=left+6 if is_heatmap else 24
                canvas=fig.add_subfigure(grid[y:y+panel['height'],left:right])
                if is_heatmap:
                    cohort={'A':'GTEx','B':'TCGA','C':'CPTAC'}[panel['letter']]
                    s10_heatmap_image=_draw_heatmap(canvas,cohort,show_colorbar=False,
                                                   show_ylabel=panel['letter']=='A')
                else:
                    panel['draw'](canvas)
            else:
                canvas=fig.add_subfigure(grid[y:y+panel['height'],x:x+panel['width']])
                panel['draw'](canvas)
        canvases.append(canvas)
    if figid=='S10' and s10_heatmap_image is not None:
        # Positions are figure fractions; the bar spans 35% of the heatmap row.
        row_fraction=3/height
        colorbar_ax=fig.add_axes([.86,1-row_fraction*.675,.012,row_fraction*.35])
        fig.colorbar(s10_heatmap_image,cax=colorbar_ax,label='Row z-score')
    qualifier='partial_' if figid=='S10' and ('S10','D') not in PANELS else ''
    stem=f'Figure{figid}_assembled_{qualifier}{FIGURE_CONTENT[figid]}'
    display(Markdown('**Assembled figure**'))
    _export_canvas(fig,stem,figid,panels,width,height,canvases,assembled=True)
    save_table(pd.DataFrame(EXPORT_RECORDS),'publication_export_manifest')


## APA analysis and source tables

Read per-sample DaPars predictions for TCGA, CPTAC, and GTEx. Where both long- and short-isoform expression fields are available, require support ≥20 in at least half of samples; otherwise require ≥50% nonmissing PDUI. Require PDUI SD ≥0.05 and range ≥0.10 before fitting. Save sample overlap, missingness, identifier mappings, event counts, binned matrices, and statistical source tables.

APA host-gene overlaps count unique mapped genes with at least one rhythmic APA event. Machinery expression panels use **mRNA fits**; splicing-machinery APA panels use **host genes carrying rhythmic APA events**. These are distinct measurements. Except for panels D and I, category-comparison asterisks denote BH q < 0.05 and ns denotes q ≥ 0.05; panels D and I use the raw-P thresholds documented below. Machinery-expression stars and displayed phases use the authoritative gene membership including its amplitude gate; the color scale shows stored BH q. Polar machinery plots label the highest-amplitude gene; the complete gene identities and fits are saved in the source table. Panel D compares the TCGA ∪ CPTAC union with GTEx. A tumor gene counts as rhythmic if it passes the same mRNA threshold in either tumor cohort; count each gene once and retain the tested category denominator. Each bar is the percentage of tested category genes passing the authoritative mRNA rhythmicity rule (BH q < 0.05 and peak-to-trough amplitude > 0.1 in TMM log2 CPM). For each category, use the 2×2 table `[[union rhythmic, union tested − rhythmic], [GTEx rhythmic, GTEx tested − rhythmic]]` in a two-sided Fisher exact test. Bracket annotations use raw P values: * P < 0.05; ** P < 0.01; *** P < 0.001; **** P < 0.0001 (most stringent threshold); comparisons with P ≥ 0.05 have no bracket or annotation. No background normalization or ER calculation is performed. These annotations are not adjusted for multiple comparisons; existing BH q columns are retained for the other documented comparisons. Other category-comparison panels retain their documented Fisher tests. Phase-lag plots retain event-level weighting, including separate observations from both cohorts for shared host genes.

GO enrichment uses the original whole-GOA annotated UniProt-protein background, across all GO namespaces, for every cohort. Foregrounds are the current rhythmic APA host genes mapped into this universe. Retain the corrected inclusive hypergeometric upper tail P(X ≥ observed overlap), computed with sf(k − 1), and exclude NOT-qualified BP memberships. Panel H filters on raw P < 0.01, ranks terms by ascending P (GO ID breaks ties), and displays at most 20 terms per cohort. Columns are the union of these selected terms, ordered globally by mean −log10(P) across the selected-cohort matrix, with absent entries scored as zero and the display score capped at 22. Higher mean significance runs left-to-right to match the reference; a dot is shown only when the term is in that cohort’s selected top 20. Dot size and color represent −log10(raw P), capped at 22 for display only. Restore the gray background, white grid, red scale, black dot edges, and alpha 0.8. Areas follow the original proportional sizing with a uniform reduction only if needed to avoid collisions at the publication width. Panel H uses a 4 × 2 module horizontal dot plot to fit the terms at the global ≥7 pt font size. BH q values remain in the source tables for reference but do not determine this panel’s selection or color. Reference mapping is built from the supplied GO annotations rather than importing legacy enrichment results.


Panel I compares TCGA ∪ CPTAC with GTEx for splicing-machinery host genes. In each cohort, a gene is rhythmic if any of its tested APA events has nominal P < 0.05. The tumor denominator is the union of tested host genes in TCGA and CPTAC; its numerator is the union of rhythmic host genes. Each gene counts once. The GTEx denominator/numerator are its own tested/rhythmic host genes. Compare the resulting rhythmic/nonrhythmic counts with a two-sided Fisher exact test. Brackets use raw P: * < 0.05, ** < 0.01, *** < 0.001, **** < 0.0001; omit nonsignificant labels and brackets. This panel measures APA-event rhythmicity, not rhythmic mRNA expression. Its source table is `Figure4I_union_gtex_splicing_host_apa_source.tsv`.


In [ ]:
# Cell 4: Embedded APA analysis and panel-drawing functions
from collections import OrderedDict

from pathlib import Path

import re

import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

from matplotlib.colors import TwoSlopeNorm

from scipy import stats

from scipy.stats import hypergeom

FONT["title"] = max(9, FONT.get("title", 9))

APA, PDUI, APA_META, OVERLAP, APA_QC, APA_GO = {}, {}, {}, {}, {}, {}

MACH_EXPR = pd.DataFrame()

APA_MACHINERY = OrderedDict({
    "CPSF complex": ["CPSF1","CPSF2","CPSF3","CPSF4","CPSF4L","FIP1L1","WDR33","CPSF6","CPSF7"],
    "CSTF complex": ["CSTF1","CSTF2","CSTF2T","CSTF3"],
    "CFIm complex": ["NUDT21","CPSF6","CPSF7"],
    "CFIIm complex": ["CLP1","PCF11"],
    "Poly(A) polymerases": ["PAPOLA","PAPOLB","PAPOLG","PAPD7","TENT2","TENT4A","TENT4B"],
    "PABPs": ["PABPN1","PABPC1","PABPC3","PABPC4","PABPC5"],
    "Symplekin/accessory": ["SYMPK","RBBP6","PAXBP1","NUDT16"],
})

SPLICING_MACHINERY = OrderedDict({
    "Core Spliceosome": ["SNRPA","SNRPB","SNRPC","SNRPD1","SNRPD2","SNRPD3","SNRPE","SNRPF","SNRPG","SNRPA1","SNRPB2","SF3A1","SF3A2","SF3A3","SF3B1","SF3B2","SF3B3","SF3B4","SF3B5","SF3B6","PRPF3","PRPF4","PRPF6","PRPF8","PRPF31","SNRNP200","EFTUD2","DDX23","PRPF19","CDC5L","PLRG1","BCAS2","CWC15","SYF2"],
    "SR Proteins": ["SRSF1","SRSF2","SRSF3","SRSF4","SRSF5","SRSF6","SRSF7","SRSF8","SRSF9","SRSF10","SRSF11","SRSF12","TRA2A","TRA2B"],
    "hnRNP Proteins": ["HNRNPA1","HNRNPA2B1","HNRNPC","HNRNPD","HNRNPF","HNRNPH1","HNRNPH2","HNRNPK","HNRNPL","HNRNPM","HNRNPU","PTBP1","PTBP2"],
    "Other Splicing Regulators": ["RBFOX1","RBFOX2","MBNL1","MBNL2","NOVA1","NOVA2","QKI","CELF1","CELF2","ESRP1","ESRP2","RBM4","RBM5","RBM10","RBM25","PRPF40A","U2AF1","U2AF2","SRRM1","SRRM2"],
})

MACH_COLORS = {"CPSF complex":"#1f77b4","CSTF complex":"#ff7f0e","CFIm complex":"#2ca02c","CFIIm complex":"#d62728","Poly(A) polymerases":"#9467bd","PABPs":"#8c564b","Symplekin/accessory":"#7f7f7f"}

SPLICE_COLORS = dict(zip(SPLICING_MACHINERY, ["#1f77b4","#ff7f0e","#2ca02c","#9467bd"]))

def _unversion(x):
    return str(x).split("|")[0].split(".")[0]

def _sample_file(root, sample):
    d = root / sample
    hits = sorted(d.glob("*_result_All_Prediction_Results.txt"))
    return hits[0] if hits else None

def _raw_apa(cohort):
    """Build a PDUI matrix from individual DaPars predictions, only phase samples.

    The column-by-column construction avoids a many-file long-table concat;
    ``usecols`` keeps reads constrained to the event metadata and three
    necessary per-sample measures.
    """
    root = DATA_DIR / "polyAPA" / cohort / "results"
    record_input(root)
    available = {p.name: p for p in root.iterdir() if p.is_dir()}
    raw_samples = [s for s in PHASE.index.unique() if s in available]
    samples = [(raw, canonical_sample(raw)) for raw in raw_samples]
    canonical = [c for _,c in samples]
    if len(canonical) != len(set(canonical)):
        raise AssertionError(f"{cohort}: duplicate raw DaPars samples after canonical_sample()")
    series, longs, shorts, metas, qrows = [], [], [], [], []
    seen_events=set(); selected_sites=[]
    wanted = ["Gene","Predicted_Proximal_APA","Loci","A_1_long_exp","A_1_short_exp","A_1_PDUI"]
    for raw_sample, sample in samples:
        f = _sample_file(root, raw_sample)
        if f is None: continue
        record_input(f)
        x = pd.read_csv(f, sep="\t", usecols=lambda c: c in wanted, low_memory=False)
        if "Gene" not in x or "A_1_PDUI" not in x: continue
        x = x.drop_duplicates("Gene", keep="first").set_index("Gene")
        series.append(pd.to_numeric(x["A_1_PDUI"], errors="coerce").rename(sample))
        lx = x["A_1_long_exp"] if "A_1_long_exp" in x else pd.Series(np.nan,index=x.index)
        sx = x["A_1_short_exp"] if "A_1_short_exp" in x else pd.Series(np.nan,index=x.index)
        longs.append(pd.to_numeric(lx, errors="coerce").rename(sample))
        shorts.append(pd.to_numeric(sx, errors="coerce").rename(sample))
        mm = x.reset_index()[[c for c in ["Gene","Predicted_Proximal_APA","Loci"] if c in x.reset_index()]]
        new=mm.loc[~mm.Gene.isin(seen_events)]
        metas.append(new)
        seen_events.update(new.Gene)
        site_rows=mm.loc[mm.Gene.str.startswith(('ENST00000395794.2|','ENST00000965092.1|'))].copy()
        site_rows['sample']=sample
        selected_sites.append(site_rows)
        qrows.append((raw_sample, sample, len(x), int(x["A_1_PDUI"].notna().sum())))
    if not series: raise RuntimeError(f"No aligned DaPars prediction files for {cohort}")
    mat = pd.concat(series, axis=1, copy=False)
    # An event can be absent from the first sample, so collect every file's
    # metadata and retain the first valid locus/proximal call in sorted sample
    # order (the sample loop follows the deterministic PHASE index order).
    meta = pd.concat(metas, ignore_index=True).rename(columns={"Gene":"event","Predicted_Proximal_APA":"proximal_apa","Loci":"loci"})
    meta = meta.sort_values("event", kind="stable").groupby("event",as_index=False).first()
    meta["transcript"] = meta["event"].map(_unversion)
    t2 = T2G.copy(); t2["transcript"] = t2["transcript"].map(_unversion)
    meta = meta.merge(t2[["transcript","ensg","gene"]].drop_duplicates("transcript"), on="transcript", how="left")
    meta["gene"] = meta["gene"].fillna(meta["event"].astype(str).str.split("|").str[1]).str.upper()
    meta = meta.drop_duplicates("event").set_index("event").reindex(mat.index)
    # Legacy criterion is read support in half the samples when the DaPars
    # expression fields are informative; chromosome-aggregated inputs without
    # usable read fields fall back to PDUI prevalence.
    prev = mat.notna().mean(axis=1)
    sd, rng = mat.std(axis=1, skipna=True), mat.max(axis=1)-mat.min(axis=1)
    long_mat, short_mat = pd.concat(longs,axis=1), pd.concat(shorts,axis=1)
    has_reads = (long_mat.fillna(0).to_numpy()>0).any() and (short_mat.fillna(0).to_numpy()>0).any()
    if has_reads:
        min_read_samples=max(1,int(np.ceil(.5*mat.shape[1])))
        expr_ok = ((long_mat >= 20).sum(axis=1) >= min_read_samples) & ((short_mat >= 20).sum(axis=1) >= min_read_samples)
    else:
        expr_ok = prev >= .50
    keep = expr_ok & (sd >= .05) & (rng >= .10)
    out = mat.loc[keep].copy()
    APA_QC[cohort] = pd.DataFrame({"metric":["raw_events","phase_matched_samples","read_support_min_samples","prevalence_pass","variable_pass"],"value":[len(mat),out.shape[1],max(1,int(np.ceil(.5*mat.shape[1]))),int((prev>=.5).sum()),len(out)]})
    sites=pd.concat(selected_sites,ignore_index=True)
    save_table(sites,f'{cohort}_prespecified_event_site_calls')
    meta['proximal_apa']=pd.to_numeric(meta['proximal_apa'],errors='coerce').astype(float)
    for event,g in sites.groupby('Gene'):
        if event in meta.index:
            prox=pd.to_numeric(g.Predicted_Proximal_APA,errors='coerce').dropna()
            if len(prox):meta.loc[event,'proximal_apa']=float(prox.median())
    return out, meta.loc[out.index], pd.DataFrame(qrows, columns=["raw_sample","sample","raw_events","pdui_nonmissing"])

def _go_resources():
    """Project identifier/GO mapping and the original whole-GOA protein universe."""
    base=DATA_DIR / 'enrichment'; info=base/'human_info.txt'; gpa=base/'goa_human.gpa'; names=base/'go_name.txt'
    for p in (info,gpa,names):record_input(p)
    ensg_to_up={}
    with info.open() as h:
        for line in h:
            z=line.strip()[1:].split('|')
            if len(z)==3:
                for source in z[:2]:ensg_to_up[source.split('.')[0]]=z[2]
    bp_name={}
    with names.open() as h:
        for line in h:
            z=line.rstrip('\n').split('\t')
            if len(z)>=3 and z[2].strip().lower()=='biological process':bp_name[z[0]]=z[1]
    up_terms={};whole_proteome=set()
    with gpa.open() as h:
        for line in h:
            if line.startswith('!'):continue
            z=line.rstrip('\n').split('\t')
            if len(z)>3:
                # Legacy universe includes all GOA accessions, across GO namespaces.
                whole_proteome.add(z[1].strip())
                if z[3] in bp_name and 'NOT' not in z[2].split('|'):
                    up_terms.setdefault(z[1].strip(),set()).add(z[3])
    save_table(pd.DataFrame({'uniprot':sorted(whole_proteome)}),'apa_go_whole_proteome_background')
    print('GO whole-proteome background:',len(whole_proteome),'annotated UniProt proteins')
    return ensg_to_up,up_terms,bp_name,whole_proteome


def _bh(p):
    p=np.asarray(p,float); out=np.full(len(p),np.nan); ok=np.isfinite(p)
    if ok.any():
        x=p[ok]; order=np.argsort(x); rank=np.empty(len(x),int); rank[order]=np.arange(1,len(x)+1)
        q=np.minimum.accumulate((x[order]*len(x)/np.arange(1,len(x)+1))[::-1])[::-1]; v=np.empty(len(x)); v[order]=np.minimum(q,1); out[ok]=v
    return out

def _category_map(groups):
    out={}
    for category,genes in groups.items():
        for gene in genes:out.setdefault(gene,category)
    return out

def _go_enrichment(foreground,background,resources):
    mapping,annotation,names,whole_proteome=resources
    bg=set(whole_proteome)
    fg={mapping[x] for x in foreground if x in mapping} & bg
    tested={mapping[x] for x in background if x in mapping} & bg
    term_members={}
    for accession,terms in annotation.items():
        if accession in bg:
            for term in terms:term_members.setdefault(term,set()).add(accession)
    rows=[]
    for term,members in term_members.items():
        if len(members)<5:continue
        overlap=len(fg & members)
        # Retain the corrected inclusive upper tail, evaluated without 1-CDF cancellation.
        p=stats.hypergeom.sf(overlap-1,len(bg),len(members),len(fg))
        rows.append(dict(term=term,term_name=names[term],overlap=overlap,term_background=len(members),
                         foreground_n=len(fg),background_n=len(bg),p=p,
                         background_type='whole_GOA_annotated_proteome',tested_host_n=len(tested)))
    out=pd.DataFrame(rows)
    out['q']=_bh(out.p) if len(out) else pd.Series(dtype=float)
    return out.sort_values(['q','p','term']) if len(out) else out


def _axial_stats(values,rng):
    values=np.asarray(values,float);values=values[np.isfinite(values)]
    axis,r2=phase_axis(values)
    result=dict(n=len(values),axis_h=axis,R2=r2)
    if len(values)<3:return dict(result,p=np.nan,ci_low=np.nan,ci_high=np.nan)
    # Chunk simulations to bound memory while retaining all event observations.
    boots=[];null=[];angles=np.exp(2j*values*np.pi/12)
    for start in range(0,1000,50):
        boots.extend(np.abs(angles[rng.integers(0,len(values),(min(50,1000-start),len(values)))].mean(axis=1)))
    for start in range(0,5000,50):
        null.extend(np.abs(np.exp(1j*rng.uniform(0,2*np.pi,(min(50,5000-start),len(values)))).mean(axis=1)))
    return dict(result,p=(1+np.sum(np.asarray(null)>=r2))/5001,ci_low=np.quantile(boots,.025),ci_high=np.quantile(boots,.975))

def _prepare_comparisons():
    global MACH_EXPR
    cmap=_category_map(APA_MACHINERY)
    rows=[]
    for c in COHORTS:
        z=GENE[c].loc[GENE[c].index.intersection(cmap)].copy()
        z['gene']=z.index;z['category']=z.gene.map(cmap);z['cohort']=c;rows.append(z.reset_index(drop=True))
    MACH_EXPR=pd.concat(rows,ignore_index=True)
    save_table(MACH_EXPR,'apa_machinery_expression_fits')
    category_rows=[]
    for kind,groups in [('machinery',APA_MACHINERY),('splicing',SPLICING_MACHINERY)]:
        for pair,tumors in [('pooled',['TCGA','CPTAC']),('TCGA',['TCGA']),('CPTAC',['CPTAC'])]:
            for category,genes in groups.items():
                if kind=='machinery':
                    tested=set(genes)&set(MACH_EXPR.gene)
                    a=set().union(*(set(significant(GENE[c]).index)&tested for c in tumors))
                    b=set(significant(GENE['GTEx']).index)&tested
                    nt=nn=len(tested)
                else:
                    def hosts(c, sig=False):
                        ids=significant(APA[c]).index if sig else APA[c].index
                        return set(APA_META[c].loc[ids,'gene'].dropna())&set(genes)
                    test_t=set().union(*(hosts(c) for c in tumors));test_n=hosts('GTEx')
                    a=set().union(*(hosts(c,True) for c in tumors));b=hosts('GTEx',True)
                    nt,nn=len(test_t),len(test_n)
                odds,p=stats.fisher_exact([[len(a),nt-len(a)],[len(b),nn-len(b)]], alternative='two-sided') if nt and nn else (np.nan,np.nan)
                category_rows.append(dict(kind=kind,pair=pair,category=category,tumor_tested=nt,normal_tested=nn,
                                          tumor_rhythmic=len(a),normal_rhythmic=len(b),tumor_pct=100*len(a)/nt if nt else np.nan,
                                          normal_pct=100*len(b)/nn if nn else np.nan,odds=odds,p=p))
    tests=pd.DataFrame(category_rows)
    tests['q']=tests.groupby('kind')['p'].transform(lambda x:_bh(x))
    APA_QC['category_tests']=tests
    save_table(tests,'machinery_and_splicing_category_tests')
    mapping_qc=[]
    for c in COHORTS:
        mapped=significant(APA[c]).join(APA_META[c][['gene','ensg']]).copy()
        gene_fit=significant(GENE_ALL[c])
        mapped['mapping_status']=np.select(
            [mapped.ensg.isna(), ~mapped.ensg.isin(gene_fit.index)],
            ['unmapped_apa_event','host_not_rhythmic_gene'],
            default='matched_rhythmic_host',
        )
        mapped['event']=mapped.index;mapped['cohort']=c
        mapping_qc.append(mapped[['cohort','event','gene','ensg','mapping_status']])
    mapping_qc=pd.concat(mapping_qc,ignore_index=True)
    APA_QC['host_expression_mapping']=mapping_qc
    save_table(mapping_qc,'apa_rhythmic_host_expression_mapping_qc')
    lag_stats=[]
    for tumor in ['TCGA','CPTAC']:
        def hostset(c):return set(APA_META[c].loc[significant(APA[c]).index,'ensg'].dropna())
        tg,ng=hostset(tumor),hostset('GTEx')
        groups={'tumor_only':tg-ng,'shared':tg&ng,'normal_only':ng-tg}
        bg=set(APA_META[tumor].loc[APA[tumor].index,'ensg'].dropna())|set(APA_META['GTEx'].loc[APA['GTEx'].index,'ensg'].dropna())
        odds,p=stats.fisher_exact([[len(tg&ng),len(tg-ng)],[len(ng-tg),len(bg-tg-ng)]],alternative='greater')
        count=pd.DataFrame([dict(set=k,genes=len(v)) for k,v in groups.items()])
        expr=[]
        for c in [tumor,'GTEx']:
            z=significant(APA[c]).join(APA_META[c][['gene','ensg','transcript']])
            gene_fit=significant(GENE_ALL[c])
            z=z.dropna(subset=['ensg']).loc[lambda x:x.ensg.isin(gene_fit.index)].copy()
            z['event']=z.index;z['cohort']=c
            z['gene_phase']=gene_fit.loc[z.ensg,'phase'].to_numpy()
            z['gene_q']=gene_fit.loc[z.ensg,'q'].to_numpy()
            z['set']=[next(k for k,v in groups.items() if g in v) for g in z.ensg]
            z['lag_h']=circular_lag(z.phase,z.gene_phase)
            expr.append(z.reset_index(drop=True))
        expression=pd.concat(expr,ignore_index=True)
        OVERLAP[f'{tumor}_GTEx']=dict(counts=count,gene_sets=groups,expression=expression,odds=odds,p=p)
        save_table(count,f'apa_host_overlap_{tumor.lower()}_vs_gtex')
        save_table(expression,f'apa_expression_phase_pairs_{tumor.lower()}_vs_gtex')
        for k in groups:
            z=expression.loc[expression['set']==k,'lag_h']%24
            row=_axial_stats(z,np.random.default_rng(SEED+len(lag_stats)))
            lag_stats.append(dict(tumor=tumor,set=k,**row))
    overlap_tests=pd.DataFrame([dict(tumor=c,p=OVERLAP[f'{c}_GTEx']['p'],odds=OVERLAP[f'{c}_GTEx']['odds']) for c in ['TCGA','CPTAC']])
    overlap_tests['q']=_bh(overlap_tests.p)
    APA_QC['overlap_tests']=overlap_tests;save_table(overlap_tests,'apa_overlap_fisher_tests')
    lag_stats=pd.DataFrame(lag_stats);lag_stats['q']=_bh(lag_stats.p)
    APA_QC['lag_statistics']=lag_stats;save_table(lag_stats,'apa_expression_lag_axial_statistics')
    amp=[]
    for column in ['amplitude','relative_amplitude']:
        for c in ['TCGA','CPTAC']:
            u,p=stats.mannwhitneyu(significant(APA[c])[column],significant(APA['GTEx'])[column],alternative='two-sided')
            amp.append(dict(measure=column,tumor=c,U=u,p=p))
    amp=pd.DataFrame(amp);amp['q']=_bh(amp.p);APA_QC['amplitude_tests']=amp;save_table(amp,'apa_amplitude_mann_whitney_tests')

def run_apa_analysis():
    resources=_go_resources();qc=[]
    for cohort in COHORTS:
        print('Reading and fitting APA:',cohort,flush=True)
        matrix,meta,sampleqc=_raw_apa(cohort)
        fit=harmonic_fit(matrix)
        # APA events use the prespecified nominal p-value criterion; mRNA
        # rhythmicity remains supplied by the authoritative GENE fit table.
        fit['is_rhythmic']=fit['p'] < P_THRESHOLD
        APA[cohort]=fit;PDUI[cohort]=matrix.loc[fit.index];APA_META[cohort]=meta.loc[fit.index]
        save_table(fit.rename_axis('event').reset_index(),f'{cohort}_apa_harmonic_fits')
        save_table(meta.rename_axis('event').reset_index(),f'{cohort}_apa_event_metadata')
        save_table(sampleqc,f'{cohort}_apa_sample_qc')
        save_table(APA_QC[cohort],f'{cohort}_apa_filter_qc')
        qc.append(dict(cohort=cohort,samples=matrix.shape[1],tested=len(fit),rhythmic_p05=len(significant(fit)),
                       mapped_events=int(meta.loc[fit.index,'ensg'].notna().sum()),missing_fraction=float(matrix.isna().to_numpy().mean())))
        APA_GO[cohort]=_go_enrichment(set(meta.loc[significant(fit).index,'ensg'].dropna()),set(meta.loc[fit.index,'ensg'].dropna()),resources)
        save_table(APA_GO[cohort],f'{cohort}_apa_go_enrichment')
        print(cohort,'APA:',qc[-1],flush=True)
    _prepare_comparisons()
    save_table(pd.DataFrame(qc),'apa_summary_qc');display(pd.DataFrame(qc))
    display(APA_QC['overlap_tests']);display(APA_QC['lag_statistics'])

CAT_SHORT={'CPSF complex':'CPSF','CSTF complex':'CSTF','CFIm complex':'CFIm','CFIIm complex':'CFIIm','Poly(A) polymerases':'Poly(A)','PABPs':'PABPs','Symplekin/accessory':'Accessory',
           'Core Spliceosome':'Core spliceosome','SR Proteins':'SR proteins','hnRNP Proteins':'hnRNPs','Other Splicing Regulators':'Other regulators'}

def _q_label(q):
    return "<1e-300" if q == 0 else (f"={q:.2g}" if np.isfinite(q) else "=NA")

def _draw_phase(canvas):
    ax=canvas.add_subplot(111,projection='polar')
    for c,cfg in COHORTS.items():
        ph=significant(APA[c]).phase;theta,density=circular_density(ph)
        ax.plot(theta,density,color=cfg['color'],lw=1,label=cfg['short']);ax.fill(theta,density,color=cfg['color'],alpha=.12)
    style_polar(ax);ax.set_title('APA acrophases',y=1.2,pad=7)
    handles,labels=ax.get_legend_handles_labels()
    canvas.legend(handles,labels,loc='outside lower center',ncol=3,handlelength=1.1,columnspacing=.8,handletextpad=.4)

def _draw_amplitude(canvas,column,raw_p=False):
    ax=canvas.subplots();values=[significant(APA[c])[column].to_numpy() for c in COHORTS]
    violin=ax.violinplot(values,showmedians=True,showextrema=False)
    for artist,c in zip(violin['bodies'],COHORTS):artist.set_facecolor(COHORTS[c]['color']);artist.set_alpha(.75)
    violin['cmedians'].set_color('0.2');violin['cmedians'].set_linewidth(.8)
    ax.set_xticks([1,2,3],list(COHORTS));ax.set_ylabel('Amplitude / PDUI SD' if column=='relative_amplitude' else 'PDUI half-amplitude')
    ax.set_title('Relative APA amplitude' if column=='relative_amplitude' else 'APA amplitude')
    tests=APA_QC['amplitude_tests'].query('measure == @column')
    text='\n'.join(f'{r.tumor}–GTEx '+(f'$p${_q_label(r.p)}' if raw_p else f'$q${_q_label(r.q)}') for r in tests.itertuples())
    ax.text(.98,.98,text,transform=ax.transAxes,ha='right',va='top',fontsize=FONT['annotation'])

def _draw_overlap(canvas,tumor,raw_p=False):
    ax=canvas.subplots();d=OVERLAP[f'{tumor}_GTEx']['counts'];colors=[GROUP_COLORS[x] for x in ['tumor_only','shared','normal_only']]
    bars=ax.bar(['Tumor\nonly','Shared','Normal\nonly'],d.genes,color=colors)
    for b,n in zip(bars,d.genes):ax.text(b.get_x()+b.get_width()/2,n,str(n),ha='center',va='bottom',fontsize=FONT['annotation'])
    test=APA_QC['overlap_tests'].set_index('tumor').loc[tumor]
    stat_label=f'$p${_q_label(test.p)}' if raw_p else f'$q${_q_label(test.q)}'
    ax.set_title(f'{tumor} vs GTEx\nFisher {stat_label}');ax.set_ylabel('Rhythmic APA host genes');ax.margins(y=.15)

def _draw_category(canvas,kind,pair,raw_p=False,show_ns=True):
    ax=canvas.subplots();d=APA_QC['category_tests'];d=d[(d.kind==kind)&(d.pair==pair)].copy()
    if raw_p:
        d['p_significance_annotation']=d.p.map(lambda p:_p_value_stars(p) if p < .05 else '')
        save_table(d,'Figure4I_union_gtex_splicing_host_apa_source')
    x=np.arange(len(d));tumor='TCGA ∪ CPTAC' if pair=='pooled' else pair
    ax.bar(x-.18,d.tumor_pct,.36,color='#D55E00' if pair!='CPTAC' else COHORTS['CPTAC']['color'],label=tumor)
    ax.bar(x+.18,d.normal_pct,.36,color=COHORTS['GTEx']['color'],label='GTEx')
    for xx,row in zip(x,d.itertuples()):
        if raw_p:
            if not row.p_significance_annotation:
                continue
            y=max(row.tumor_pct,row.normal_pct)+5
            ax.plot([xx-.18,xx-.18,xx+.18,xx+.18],[y,y+2,y+2,y],color='0.25',linewidth=.6)
            ax.text(xx,y+3,row.p_significance_annotation,ha='center',va='bottom',fontsize=FONT['annotation'])
        else:
            if not show_ns and not row.q < Q_THRESHOLD:
                continue
            label='*' if row.q<Q_THRESHOLD else 'ns'
            ax.text(xx,max(row.tumor_pct,row.normal_pct)+3,label,ha='center',fontsize=FONT['annotation'])
    ax.set_xticks(x,[CAT_SHORT[c] for c in d.category],rotation=30,ha='right');ax.set_ylim(0,140);ax.set_yticks([0,25,50,75,100])
    ax.set_ylabel('Genes with rhythmic\nAPA events (%)' if raw_p else 'Rhythmic genes (%)')
    ax.legend(loc='upper left',ncol=1 if raw_p else 2)
    ax.set_title('Splicing-host APA\nTCGA ∪ CPTAC vs GTEx' if raw_p else
                 ('APA machinery expression' if kind=='machinery' else f'Splicing-hosted APA: {pair}'))


def _p_value_stars(p):
    if not np.isfinite(p):
        return 'NA'
    return '****' if p < .0001 else '***' if p < .001 else '**' if p < .01 else '*' if p < .05 else 'ns'


def _draw_machinery_rhythmic_percentage(canvas):
    """Tumor-union/GTEx category percentages, with raw two-sided Fisher P brackets."""
    d=APA_QC['category_tests'].query("kind == 'machinery' and pair == 'pooled'").copy()
    d['p_significance_annotation']=d.p.map(lambda p: _p_value_stars(p) if p < .05 else '')
    save_table(d,'Figure4D_union_gtex_apa_machinery_expression_source')
    ax=canvas.subplots();x=np.arange(len(d));width=.28;offset=.15
    ax.bar(x-offset,d.tumor_pct,width=width,color=COHORTS['TCGA']['color'],label='TCGA ∪ CPTAC')
    ax.bar(x+offset,d.normal_pct,width=width,color=COHORTS['GTEx']['color'],label='GTEx')
    for xx,row in zip(x,d.itertuples()):
        if not row.p_significance_annotation:
            continue
        y=max(row.tumor_pct,row.normal_pct)+5
        ax.plot([xx-offset,xx-offset,xx+offset,xx+offset],
                [y,y+2,y+2,y],color='0.25',linewidth=.6)
        ax.text(xx,y+3,row.p_significance_annotation,ha='center',va='bottom',
                fontsize=FONT['annotation'])
    ax.set_xticks(x,[CAT_SHORT[c] for c in d.category],rotation=30,ha='right')
    ax.set_ylim(0,130);ax.set_yticks([0,25,50,75,100])
    ax.set_ylabel('Rhythmic genes (%)')
    ax.set_title('APA machinery\nTCGA ∪ CPTAC vs GTEx')
    ax.legend(loc='upper right',ncol=1,fontsize=FONT['legend'],
              handlelength=1.1,columnspacing=1.1,handletextpad=.4)


def _machinery_legend(canvas, columns=2, location='outside lower center'):
    from matplotlib.lines import Line2D
    handles=[Line2D([],[],marker='o',linestyle='',markersize=3.5,
                    color=MACH_COLORS[cat],label=CAT_SHORT[cat]) for cat in APA_MACHINERY]
    return canvas.legend(handles=handles,loc=location,ncol=columns,
                         handlelength=.6,handletextpad=.35,columnspacing=.65,
                         borderpad=.1,labelspacing=.2)


def _draw_machinery_polar(canvas,cohort,compact=False,show_legend=True):
    if compact:
        # Explicit positions reserve only the shared key area, not a side legend.
        box=[.13,.24,.76,.55] if show_legend else [.13,.12,.76,.66]
        ax=canvas.add_axes(box,projection='polar')
    else:
        grid=canvas.add_gridspec(1,2,width_ratios=[1,.7])
        ax=canvas.add_subplot(grid[0,0],projection='polar')
        key_ax=canvas.add_subplot(grid[0,1]);key_ax.axis('off')
    d=MACH_EXPR[(MACH_EXPR.cohort==cohort)&MACH_EXPR.is_rhythmic.fillna(False)]
    for cat in APA_MACHINERY:
        z=d[d.category==cat]
        if len(z):ax.scatter(z.phase*np.pi/12,z.amplitude,s=12,color=MACH_COLORS[cat],label=CAT_SHORT[cat])
    for _,r in d.nlargest(1,'amplitude').iterrows():
        ax.annotate(r.gene,(r.phase*np.pi/12,r.amplitude),xytext=(2,5),textcoords='offset points',fontsize=FONT['annotation'])
    style_polar(ax)
    ax.tick_params(pad=1)
    if compact:
        canvas.text(.5,.99,f'{cohort} (n={len(d)})',ha='center',va='top',fontsize=FONT['title'])
    else:
        ax.set_title(f'{cohort} (n={len(d)})',y=1.2,pad=7)
    ax.yaxis.set_major_locator(plt.MaxNLocator(3))
    if compact and show_legend:
        _machinery_legend(canvas,columns=4,location='lower center')
    elif not compact:
        handles,labels=ax.get_legend_handles_labels()
        key_ax.legend(handles,labels,loc='center left',handletextpad=.2,borderaxespad=0)


def _draw_gene_scatter(canvas):
    axes=canvas.subplots(1,2,sharex=True,sharey=True);data=OVERLAP['TCGA_GTEx']['expression']
    # Dark blue and higher opacity distinguish normal-only events from shared events.
    scatter_colors={'tumor_only':'#D55E00','shared':'#009E73','normal_only':'#3046A5'}
    for ax,c in zip(axes,['TCGA','GTEx']):
        d=data[data.cohort==c]
        for key,name in [('tumor_only','Tumor only'),('shared','Shared'),('normal_only','Normal only')]:
            z=d[d['set']==key];ax.scatter(z.gene_phase,z.phase,s=3,alpha=.7,c=scatter_colors[key],label=name,rasterized=True)
        ax.plot([0,24],[0,24],ls='--',lw=.5,c='gray');ax.set(xlim=(0,24),ylim=(0,24),xticks=[0,6,12,18,24],yticks=[0,6,12,18,24],xlabel='Host-expression phase (h)',title=f'{c} (n={len(d):,})')
    axes[0].set_ylabel('APA phase (h)')
    handles,labels=axes[1].get_legend_handles_labels()
    legend=canvas.legend(handles,labels,loc='outside lower center',ncol=3,markerscale=2,
                         handletextpad=.2,columnspacing=.6)
    for handle in legend.legend_handles:handle.set_alpha(1)


def _select_go_plot_terms():
    selected=[]
    for cohort in COHORTS:
        d=APA_GO[cohort].loc[APA_GO[cohort].p < GO_P_THRESHOLD].sort_values(['p','term']).head(GO_MAX_TERMS_PER_COHORT).copy()
        d['cohort']=cohort
        d['cohort_p_rank']=np.arange(1,len(d)+1)
        selected.append(d)
    return pd.concat(selected,ignore_index=True)


def _draw_go(canvas):
    import textwrap
    selected=_select_go_plot_terms()
    ax=canvas.subplots()
    if selected.empty:
        ax.text(.5,.5,'No GO BP enrichment\nat raw $p$ < 0.01',ha='center',va='center')
        ax.axis('off');save_table(selected,'Figure4H_go_plot_data');return
    # Original comparison averages the capped scores in the selected-cohort matrix,
    # assigning zero to absent cohort entries. Stronger terms run left-to-right in
    # the horizontal presentation, matching the supplied reference image.
    selected['display_score']=-np.log10(selected.p.clip(lower=10.**(-GO_DISPLAY_SCORE_MAX)))
    pivot=selected.pivot(index='term',columns='cohort',values='display_score').reindex(columns=list(COHORTS)).fillna(0)
    order=pivot.mean(axis=1).rename('mean_display_score').reset_index().sort_values(['mean_display_score','term'])
    terms=order.term.tolist();names=selected.drop_duplicates('term').set_index('term').term_name
    ax.set_facecolor('#EAEAF2');ax.set_axisbelow(True)
    ax.grid(True,axis='both',color='white',linewidth=.5,alpha=.9)
    for spine in ax.spines.values():spine.set_visible(False)
    # Preserve the original area ∝ -log10(P) relationship while fitting the A4 grid.
    # The 30 pt²/score legacy areas are uniformly scaled only when they would collide.
    max_score=max(1.,float(selected.display_score.max()))
    column_pitch_pt=(4*MODULE_MM/25.4*72-40)/len(terms)
    area_factor=min(30.,(.82*column_pitch_pt)**2/max_score)
    rows=[];sc=None
    for i,cohort in enumerate(COHORTS):
        d=APA_GO[cohort].set_index('term').reindex(terms)
        ranks=selected.loc[selected.cohort==cohort].set_index('term').cohort_p_rank
        d['cohort_p_rank']=ranks.reindex(terms)
        d['selected_for_plot']=d.cohort_p_rank.notna()
        d['plot_order']=np.arange(len(terms));d['mean_display_score']=order.set_index('term').loc[terms,'mean_display_score']
        score=-np.log10(d.p.clip(lower=10.**(-GO_DISPLAY_SCORE_MAX)))
        d['display_score']=score.where(d.selected_for_plot,0)
        keep=d.selected_for_plot.to_numpy()
        sc=ax.scatter(np.arange(len(terms))[keep],np.full(int(keep.sum()),i),
                      s=score.to_numpy()[keep]*area_factor,c=score.to_numpy()[keep],cmap='Reds',
                      vmin=0,vmax=max_score,alpha=.8,edgecolors='black',lw=.5)
        rows.append(d.reset_index().assign(cohort=cohort))
    ax.set_xticks(range(len(terms)),[textwrap.fill(names[t],60,break_long_words=False,break_on_hyphens=False) for t in terms],rotation=90,ha='center',va='top')
    ax.set_yticks(range(len(COHORTS)),list(COHORTS));ax.invert_yaxis()
    ax.set_xlim(-.7,len(terms)-.3);ax.set_ylim(len(COHORTS)-.5,-.5)
    ax.set_title('Rhythmic APA host genes: GO BP')
    ax.tick_params(axis='both',length=0,pad=2)
    canvas.colorbar(sc,ax=ax,label=r'$-\log_{10}(p)$',location='top',shrink=.35,aspect=35,pad=.06)
    save_table(pd.concat(rows,ignore_index=True),'Figure4H_go_plot_data')
    save_table(selected,'Figure4H_go_selected_terms')


def _machinery_matrix():
    genes=list(_category_map(APA_MACHINERY));genes=[g for g in genes if g in set(MACH_EXPR.gene)]
    q=MACH_EXPR.pivot(index='gene',columns='cohort',values='q').reindex(index=genes,columns=list(COHORTS))
    ph=MACH_EXPR.pivot(index='gene',columns='cohort',values='phase').reindex(index=genes,columns=list(COHORTS))
    rhythmic=MACH_EXPR.pivot(index='gene',columns='cohort',values='is_rhythmic').reindex(index=genes,columns=list(COHORTS)).fillna(False).astype(bool)
    return q,ph,rhythmic

def _draw_machinery_heat(canvas,phase=False):
    ax=canvas.subplots();q,ph,rhythmic=_machinery_matrix()
    arr=ph.where(rhythmic).to_numpy() if phase else (-np.log10(q.clip(lower=1e-300))).clip(0,5).to_numpy()
    cmap=mpl.colormaps['twilight' if phase else 'Reds'].copy();cmap.set_bad('#EEEEEE')
    im=ax.imshow(np.ma.masked_invalid(arr),aspect='auto',cmap=cmap,vmin=0,vmax=24 if phase else 5)
    ax.set_xticks(range(3),list(COHORTS),rotation=45,ha='right')
    ax.set_yticks(range(len(q)),q.index);ax.set_title('Acrophase' if phase else 'Machinery expression')
    if not phase:
        for i in range(len(q)):
            for j in range(3):
                if rhythmic.iloc[i,j]:ax.text(j,i,'*',ha='center',va='center',fontsize=FONT['annotation'])
    canvas.colorbar(im,ax=ax,label='Phase (h)' if phase else r'$-\log_{10}(q)$ (BH; cap 5)',shrink=.45,pad=.03)
    save_table(q.rename_axis('gene').reset_index(),'machinery_expression_q_matrix');save_table(rhythmic.rename_axis('gene').reset_index(),'machinery_expression_is_rhythmic_matrix');save_table(ph.where(rhythmic).rename_axis('gene').reset_index(),'machinery_expression_phase_matrix')

def _draw_heatmap(canvas,cohort,show_colorbar=True,show_ylabel=True):
    ax=canvas.subplots();events=significant(APA[cohort]).sort_values('phase').index;mat=PDUI[cohort].loc[events]
    bins=np.floor(PHASE.loc[mat.columns]).astype(int)%24;binned=np.full((len(mat),24),np.nan)
    for hour in range(24):
        columns=mat.columns[bins==hour]
        if len(columns):binned[:,hour]=mat.loc[:,columns].mean(axis=1)
    mu=np.nanmean(binned,axis=1,keepdims=True);sd=np.nanstd(binned,axis=1,keepdims=True)
    z=np.divide(binned-mu,sd,out=np.full_like(binned,np.nan),where=sd>0)
    cmap=mpl.colormaps['bwr'].copy();cmap.set_bad('#DDDDDD')
    im=ax.imshow(np.ma.masked_invalid(z),aspect='auto',cmap=cmap,vmin=-2,vmax=2,interpolation='nearest',rasterized=True)
    ax.set_title(f'{cohort}: {len(events):,} events');ax.set_xticks([0,6,12,18,23]);ax.set_yticks([]);ax.set_xlabel('CHIRAL phase bin (h)');ax.set_ylabel('APA events by acrophase' if show_ylabel else '')
    if show_colorbar:
        canvas.colorbar(im,ax=ax,label='Row z-score',shrink=.35,pad=.025)
    for arr,name in [(binned,'mean'),(z,'zscore')]:save_table(pd.DataFrame(arr,index=events,columns=[str(h) for h in range(24)]).rename_axis('event').reset_index(),f'{cohort}_apa_binned_{name}')
    save_table(APA[cohort].loc[events].rename_axis('event').reset_index(),f'{cohort}_apa_heatmap_event_order')
    return im

def _draw_lags(canvas,tumor):
    canvas.suptitle(f'{tumor} vs GTEx: APA − expression phase (h)',fontsize=FONT['title'])
    axes=canvas.subplots(1,3,subplot_kw={'projection':'polar'});data=OVERLAP[f'{tumor}_GTEx']['expression'];statsdf=APA_QC['lag_statistics']
    for ax,key,label in zip(axes,['tumor_only','shared','normal_only'],['Tumor only','Shared','Normal only']):
        values=data.loc[data['set']==key,'lag_h']%24;theta,density=circular_density(values)
        color=GROUP_COLORS[key];ax.plot(theta,density,c=color,lw=1);ax.fill(theta,density,color=color,alpha=.18)
        s=statsdf[(statsdf.tumor==tumor)&(statsdf['set']==key)].iloc[0]
        for h in [s.axis_h,s.axis_h+12]:ax.axvline(h*np.pi/12,c='0.3',ls='--',lw=.7)
        style_polar(ax);ax.set_xticklabels(['0','+4','+8','±12','−8','−4']);ax.set_yticklabels([])
        ax.set_title(f'{label}\nn={len(values):,}',color=color,y=1.2,pad=7)
        ax.text(.5,-.4,f'R₂={s.R2:.2f}; $q$={s.q:.2g}',transform=ax.transAxes,ha='center',fontsize=FONT['annotation'])

def register_apa_panels():
    register_panel('4','A','apa_acrophase_density',_draw_phase,1,1)
    register_panel('4','B','relative_apa_amplitude',lambda c:_draw_amplitude(c,'relative_amplitude',raw_p=True),1,1)
    register_panel('4','C','tcga_gtex_apa_host_overlap',lambda c:_draw_overlap(c,'TCGA',raw_p=True),1,1)
    register_panel('4','D','union_gtex_apa_machinery_rhythmic_percentage',_draw_machinery_rhythmic_percentage,1,1)
    register_panel('4','E','gtex_apa_machinery_phases',lambda c:_draw_machinery_polar(c,'GTEx',compact=True),1,1)
    register_panel('4','F','tcga_apa_machinery_phases',lambda c:_draw_machinery_polar(c,'TCGA',compact=True,show_legend=False),1,1)
    register_panel('4','G','apa_host_expression_phases',_draw_gene_scatter,2,1)
    register_panel('4','H','apa_host_go_enrichment',_draw_go,4,2)
    register_panel('4','I','union_gtex_splicing_host_apa',lambda c:_draw_category(c,'splicing','pooled',raw_p=True),1,1)
    for letter,cohort in zip('ABC',['GTEx','TCGA','CPTAC']):register_panel('S10',letter,f'{cohort.lower()}_apa_heatmap',lambda c,g=cohort:_draw_heatmap(c,g),1,3)
    register_panel('S11','A','absolute_apa_amplitude',lambda c:_draw_amplitude(c,'amplitude'),1,1)
    register_panel('S11','B','cptac_gtex_apa_host_overlap',lambda c:_draw_overlap(c,'CPTAC'),1,1)
    register_panel('S11','C','cptac_apa_machinery_phases',lambda c:_draw_machinery_polar(c,'CPTAC'),2,1)
    register_panel('S11','D','apa_machinery_expression_rhythmicity',lambda c:_draw_machinery_heat(c,False),2,2)
    register_panel('S11','E','apa_machinery_expression_acrophases',lambda c:_draw_machinery_heat(c,True),2,2)
    register_panel('S12','A','tcga_gtex_apa_expression_lags',lambda c:_draw_lags(c,'TCGA'),4,1)
    register_panel('S12','B','cptac_gtex_apa_expression_lags',lambda c:_draw_lags(c,'CPTAC'),4,1)
    register_panel('S12','C','cptac_splicing_host_apa',lambda c:_draw_category(c,'splicing','CPTAC',show_ns=False),2,1)


In [ ]:
# Cell 5: Run APA analysis and export source tables
run_apa_analysis()
print("APA rhythmicity: nominal P <", P_THRESHOLD)
print("Panel D: TCGA union CPTAC vs GTEx; rhythmic genes (%); raw two-sided Fisher exact P values.")
display(APA_QC['category_tests'].query("kind == 'machinery' and pair == 'pooled'")[
    ['category','tumor_rhythmic','tumor_tested','normal_rhythmic','normal_tested','tumor_pct','normal_pct','p']])

_go_selected = _select_go_plot_terms()
print("Panel H: raw P <", GO_P_THRESHOLD, "; at most", GO_MAX_TERMS_PER_COHORT, "terms per cohort.")
print("Displayed GO terms per cohort:", _go_selected.groupby('cohort').size().to_dict(),
      "; unique terms:", _go_selected.term.nunique())

print("Panel I: TCGA union CPTAC vs GTEx; APA-host gene percentages; raw two-sided Fisher P.")
display(APA_QC['category_tests'].query("kind == 'splicing' and pair == 'pooled'")[
    ['category','tumor_rhythmic','tumor_tested','normal_rhythmic','normal_tested','tumor_pct','normal_pct','p']])


## miRNA, predicted targets, and the prespecified CXCL12 example

TCGA/CPTAC processed GDC read counts are normalized to CPM and transformed as `log2(CPM+1)`. GTEx uses its portal TPM matrix and RNAcentral-to-miRBase mapping, transformed as `log2(TPM+1)`. miRNA identifiers are normalized transparently for joining predicted targets; mapped/unmapped and sample-matching QC are exported.

The annotated `miRNA/targets/predicted_targets.tsv` is a processed TargetScan reference input. The reference can have an empty ENSG column: missing IDs are resolved only through unambiguous exact gene-symbol matches to the authoritative gene tables, with mapped/unmapped QC exported; subsequent target joins use ENSG. Predicted interactions are not experimental validation. Target alignment and correlation populations are rebuilt from the new rhythmicity calls. Correlation histograms use a deterministic maximum of 20,000 eligible unique pairs, and export the selected population and BH correction. The original chi-square comparisons of anti-phase/in-phase/other pair composition are retained with BH correction across the two tumor-versus-GTEx tests. These are descriptive pair-level comparisons: target pairs share features and are not independent biological replicates.

The prespecified example is **hsa-mir-135b → CXCL12, APA transcript ENST00000395794.2**. Its statistics and eligibility are recalculated without switching to a more favorable example. Any failed selection criterion is explicitly reported. The distal-UTR motif assessment uses the supplied genomic FASTA and APA coordinates; a predicted sequence match alone does not establish regulation. DaPars intervals are read as 0-based half-open and reported seed spans are GRCh38 1-based inclusive. The schematic uses the median proximal PAS across matched TCGA sample calls; per-sample PAS calls are also exported.

In [ ]:
# Cell 6: Embedded miRNA, target, and selected-example functions
"""Self-contained miRNA analysis and drawing fragment for the Fig. 4 notebook.

This file deliberately has no runtime side effects.  The enclosing notebook supplies
the shared globals documented in contract.md and calls run_mirna_analysis() followed
by register_mirna_panels().
"""
from pathlib import Path
import math
import re
import gzip
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from scipy import stats


MIRNA, MIRFIT, MIRNA_QC, TARGETS, MIRNA_PAIRS, MIRNA_CORR = {}, {}, {}, pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
CXCL12_CASE, MIRNA_PHASE_SUMMARY = {}, pd.DataFrame()


def _key(x, target=False):
    if pd.isna(x): return np.nan
    x = str(x).strip().lower().replace("_", "-")
    x = re.sub(r"\.(\d+)$", "", x)
    x = re.sub(r"-(3p|5p)$", "", x)
    return re.sub(r"^(hsa-(?:mir|let)-\d+[a-z]?)-\d+$", r"\1", x) if target else x


def _gene(x): return str(x).strip().upper() if pd.notna(x) else np.nan
def _phase_rad(hours): return np.asarray(hours, float) * 2*np.pi/24
def _safeq(p):
    p=np.asarray(p,float); out=np.full(len(p),np.nan); ok=np.isfinite(p)
    if ok.any():
        v=p[ok]; order=np.argsort(v); rank=np.arange(1,len(v)+1); q=np.minimum.accumulate((v[order]*len(v)/rank)[::-1])[::-1]; out[np.where(ok)[0][order]]=np.minimum(q,1)
    return out


def _record(path):
    return record_input(path)


def _gdc_matrix(folder):
    files=[]
    for p in sorted(Path(folder).glob("*.tsv")):
        try:
            h=pd.read_csv(p,sep="\t",nrows=0).columns.tolist()
            if h==["miRNA_ID","read_count","reads_per_million_miRNA_mapped","cross-mapped"]: files.append(p)
        except (OSError, pd.errors.ParserError): continue
    if not files: raise FileNotFoundError(f"No valid GDC miRNA files in {folder}")
    ids=None; cols=[]; crossed=None
    for p in files:
        _record(p); d=pd.read_csv(p,sep="\t"); cur=d.miRNA_ID.astype(str).to_numpy()
        if ids is None: ids=cur; crossed=np.zeros(len(ids),bool)
        if not np.array_equal(ids,cur): raise ValueError(f"miRNA order differs: {p}")
        cols.append(pd.to_numeric(d.read_count,errors="coerce").fillna(0).to_numpy())
        crossed |= d["cross-mapped"].astype(str).str.upper().eq("Y").to_numpy()
    sample_ids=[canonical_sample(p.stem) for p in files]
    if pd.Index(sample_ids).duplicated().any():
        dup=pd.Index(sample_ids)[pd.Index(sample_ids).duplicated()].unique().tolist()
        raise ValueError(f"GDC columns collapse to duplicate biological samples: {dup[:5]}")
    raw=pd.DataFrame(np.column_stack(cols),index=ids,columns=sample_ids)
    cpm=raw.div(raw.sum(axis=0).replace(0,np.nan),axis=1)*1e6
    expr=np.log2(cpm.fillna(0)+1)
    return raw,expr,pd.Series(crossed,index=raw.index)


def _gtex_matrix():
    p=DATA_DIR/"miRNA/GTEx/raw/miRNA_TPM_matrix_PORTAL_2025_03_17.txt"; m=DATA_DIR/"miRNA/GTEx/raw/rnacentral_mirbase.tsv"
    _record(p); _record(m)
    _record(DATA_DIR/"Merged/GTEx_TCGA_merged_meta.csv")
    meta=pd.read_csv(DATA_DIR/"Merged/GTEx_TCGA_merged_meta.csv",sep="\t",index_col=0)
    want=[str(x) for x in meta.index[meta.Info.eq("GTEx_Normal")]]
    base={x.rsplit("-SM-",1)[0]:x for x in want}
    header=pd.read_csv(p,sep="\t",nrows=0); idcol=header.columns[0]
    keep=[c for c in header.columns[1:] if str(c).rsplit("-SM-",1)[0] in base]
    raw=pd.read_csv(p,sep="\t",usecols=[idcol]+keep).rename(columns={idcol:"rnacentral_id"}).set_index("rnacentral_id")
    raw=raw.apply(pd.to_numeric,errors="coerce").fillna(0).rename(columns={c:canonical_sample(base[str(c).rsplit("-SM-",1)[0]]) for c in keep})
    mp=pd.read_csv(m,sep="\t",header=None,usecols=list(range(6)),names=["id","db","accession","taxon","type","name"],dtype=str)
    mp=mp[(mp.taxon.eq("9606"))&mp.name.str.startswith("hsa-",na=False)].copy()
    mp["tr"] = np.select([mp.type.eq("miRNA"),mp.type.eq("pre_miRNA")],[0,1],2); mp["ar"]=np.where(mp.accession.str.startswith("MIMAT",na=False),0,1)
    mp=mp.sort_values(["id","tr","ar","name"]).drop_duplicates("id").set_index("id").name.to_dict()
    raw.index=[mp.get(str(x),str(x)) for x in raw.index]
    raw=raw.groupby(level=0).mean()
    if raw.columns.duplicated().any():
        dup=raw.columns[raw.columns.duplicated()].unique().tolist()
        raise ValueError(f"GTEx columns collapse to duplicate biological samples: {dup[:5]}")
    raw=raw.loc[:,[x for x in raw.columns if x in PHASE.index]]
    return raw,np.log2(raw+1),pd.Series(False,index=raw.index)


def _filter(raw, expr, crossed, kind):
    n=raw.shape[1]; prevalent=int(math.ceil(.6*n))
    if kind=="GDC": keep=(raw.ge(50).sum(1)>=prevalent)&(expr.ge(1).sum(1)>=prevalent)&~crossed
    else: keep=(raw.ge(1).sum(1)>=prevalent)
    keep &= expr.std(1)>=.15; keep &= (expr.max(1)-expr.min(1))>=1
    return expr.loc[keep]


def _fit(expr):
    # APA and miRNA share the nominal-P harmonic fitting implementation.
    return harmonic_fit(expr)


def _targets():
    p=DATA_DIR/"miRNA/targets/predicted_targets.tsv"; _record(p); d=pd.read_csv(p,sep="\t")
    d["mirna_key"]=d.mirna_id.map(lambda x:_key(x,True)); d["gene"]=d.target_gene_symbol.map(_gene)
    # Some processed TargetScan references have an entirely empty ENSG column.
    # Resolve exact, unambiguous symbols against the authoritative gene tables;
    # never turn missing values into the literal string 'nan' or use fuzzy matching.
    d["ensg"]=d.target_gene_ensg.astype('string').str.replace(r"\.\d+$","",regex=True)
    d['ensg']=d.ensg.mask(d.ensg.isin(['','nan','None','<NA>']))
    reference=pd.concat([f[['gene','ensg']].reset_index(drop=True) for f in GENE_ALL.values()]).drop_duplicates()
    unambiguous=reference[~reference.gene.duplicated(False)].set_index('gene').ensg
    provided=d.ensg.notna()
    d['ensg']=d.ensg.fillna(d.gene.map(unambiguous))
    d['ensg_mapping_status']=np.select([provided,d.ensg.notna()],['provided_ensg','exact_authoritative_symbol'],default='unmapped_or_ambiguous')
    save_table(d[['mirna_key','gene','ensg','ensg_mapping_status']],'mirna_target_ensg_mapping')
    save_table(d.groupby('ensg_mapping_status',dropna=False).size().rename('reference_rows').reset_index(),'mirna_target_ensg_mapping_qc')
    return d.dropna(subset=["mirna_key","gene","ensg"]).drop_duplicates(["mirna_key","ensg"])


def _pair_table():
    rows=[]
    for cohort in COHORTS:
        mf=MIRFIT[cohort].copy();mf['mirna_key']=[_key(x,True) for x in mf.index]
        gf=GENE_ALL[cohort].reset_index(drop=True).rename(columns={'gene':'expression_gene_symbol'})
        x=TARGETS.merge(mf.reset_index(names='matrix_mirna_id'),on='mirna_key',how='inner',suffixes=('','_mir'))
        x=x.merge(gf,on='ensg',how='inner',suffixes=('_mir','_gene'))
        x['target_gene_symbol_reference']=x['gene']
        x['gene']=x.expression_gene_symbol
        x['mirna_phase']=pd.to_numeric(x.phase_mir,errors='coerce')
        x['gene_phase']=pd.to_numeric(x.phase_gene,errors='coerce')
        x['mirna_q']=pd.to_numeric(x.q_mir,errors='coerce')
        x['gene_q']=pd.to_numeric(x.q_gene,errors='coerce')
        x['mirna_p']=pd.to_numeric(x.p_mir,errors='coerce')
        x['gene_p']=pd.to_numeric(x.p_gene,errors='coerce')
        x['mirna_rhythmic']=x.mirna_p<P_THRESHOLD
        x['gene_rhythmic']=x.is_rhythmic_gene.astype(bool)
        x['both_rhythmic']=x.mirna_rhythmic & x.gene_rhythmic
        x['lag']=((x.gene_phase-x.mirna_phase+12)%24)-12
        x['phase_class']=np.where(abs(x.lag)<=PHASE_WINDOW_H,'in_phase',np.where(abs(abs(x.lag)-12)<=PHASE_WINDOW_H,'anti_phase','other'))
        x['score']=-np.log10(x.mirna_p.clip(lower=1e-300))-np.log10(x.gene_p.clip(lower=1e-300))
        x['cohort']=cohort;rows.append(x)
    return pd.concat(rows,ignore_index=True) if rows else pd.DataFrame()


def _spearman_rows(a, b):
    """Pairwise Spearman tests, with the same tie and missing-value rules as scipy."""
    a=np.asarray(a,float); b=np.asarray(b,float)
    valid=np.isfinite(a)&np.isfinite(b); n=valid.sum(axis=1)
    ar=stats.rankdata(np.where(valid,a,np.nan),axis=1,nan_policy='omit')
    br=stats.rankdata(np.where(valid,b,np.nan),axis=1,nan_policy='omit')
    with np.errstate(invalid='ignore',divide='ignore'):
        ar=ar-np.nanmean(ar,axis=1,keepdims=True)
        br=br-np.nanmean(br,axis=1,keepdims=True)
        rho=np.nansum(ar*br,axis=1)/np.sqrt(np.nansum(ar*ar,axis=1)*np.nansum(br*br,axis=1))
        rho=np.clip(rho,-1,1)
        t=rho*np.sqrt((n-2)/((1+rho)*(1-rho)))
    return n,rho,2*stats.t.sf(np.abs(t),n-2)


def _correlations(pairs):
    out=[]
    for cohort in COHORTS:
        x=pairs[pairs.cohort.eq(cohort)&pairs.both_rhythmic].copy()
        x=x[x.ensg.isin(EXPR_ENSG[cohort].index)&x.matrix_mirna_id.isin(MIRNA[cohort].index)]
        x=x.sort_values(['score','mirna_key','ensg','matrix_mirna_id'],ascending=[False,True,True,True]).drop_duplicates(['mirna_key','ensg']).head(CORRELATION_PAIR_LIMIT)
        common=[s for s in MIRNA[cohort].columns if s in EXPR[cohort].columns]
        chunks=[]
        for first in range(0,len(x),512):
            z=x.iloc[first:first+512]
            a=MIRNA[cohort].loc[z.matrix_mirna_id.tolist(),common].to_numpy(float)
            b=EXPR_ENSG[cohort].loc[z.ensg.tolist(),common].to_numpy(float)
            n,rho,p=_spearman_rows(a,b)
            d=z[['cohort','matrix_mirna_id','mirna_key','gene','ensg','lag','phase_class','score']].rename(columns={'matrix_mirna_id':'mirna_id'}).copy()
            d['n']=n;d['rho']=rho;d['p']=p
            chunks.append(d.loc[d.n>=MIN_SAMPLES])
        if chunks:
            d=pd.concat(chunks,ignore_index=True);d['q']=_safeq(d.p);out.append(d)
    return pd.concat(out,ignore_index=True) if out else pd.DataFrame()


def _series_for_event(cohort,event):
    if cohort not in PDUI or event not in PDUI[cohort].index: return pd.Series(dtype=float)
    return pd.to_numeric(PDUI[cohort].loc[event],errors="coerce")


def _faidx_fetch(fasta, chrom, start0, end0):
    """Small stdlib FASTA-index fetch (0-based, half-open); avoids a pysam requirement."""
    idx={}
    with open(str(fasta)+".fai",encoding="utf-8") as h:
        for line in h:
            n,l,o,b,w=line.rstrip("\n").split("\t")[:5]; idx[n]=(int(l),int(o),int(b),int(w))
    _,off,bases,width=idx[str(chrom)]; pieces=[]
    with open(fasta,"rb") as h:
        while start0<end0:
            line=start0//bases; col=start0%bases; take=min(end0-start0,bases-col)
            h.seek(off+line*width+col); pieces.append(h.read(take).decode("ascii")); start0+=take
    return "".join(pieces).upper()


def _meta_event(cohort,gene="CXCL12"):
    m=APA_META.get(cohort,pd.DataFrame()).copy()
    if m.empty: return None,None
    cols={str(c).lower():c for c in m.columns}; gc=next((cols[k] for k in cols if k in ("gene","gene_symbol","host_gene")),None)
    q=m[m[gc].astype(str).str.upper().eq(gene)] if gc else m.iloc[0:0]
    # The prespecified transcript is a scientific constraint, not a rank choice.
    q=q[q.index.astype(str).str.startswith("ENST00000395794.2|")]
    if q.empty: return None,None
    return str(q.index[0]),q.iloc[0]


def _cxcl12_case():
    cohort="TCGA"; mir="hsa-mir-135b"; gene="CXCL12"; event,meta=_meta_event(cohort,gene)
    mf=MIRFIT[cohort]; mi=next((i for i in mf.index if _key(i,True)==mir),None); gf=GENE[cohort]; gi=next((i for i in gf.index if _gene(i)==gene),None)
    ap=APA.get(cohort,pd.DataFrame()); ai=event if event in ap.index else None
    result=dict(cohort=cohort,mirna_id=mi,gene=gene,event=event,mirna_fit=mf.loc[mi] if mi else None,gene_fit=gf.loc[gi] if gi else None,apa_fit=ap.loc[ai] if ai else None,meta=meta,gene_display_fit=GENE_DISPLAY_FITS.get(cohort,pd.DataFrame()).loc[gi] if gi in GENE_DISPLAY_FITS.get(cohort,pd.DataFrame()).index else None)
    if mi and gi:
        common=[s for s in MIRNA[cohort].columns if s in EXPR[cohort].columns]; a=MIRNA[cohort].loc[mi,common]; b=EXPR[cohort].loc[gi,common]; ok=a.notna()&b.notna(); rho,pv=stats.spearmanr(a[ok],b[ok]); result.update(rho=rho,corr_p=pv,corr_n=int(ok.sum()))
        qrow=MIRNA_CORR[(MIRNA_CORR.cohort==cohort)&(MIRNA_CORR.mirna_key==mir)&(MIRNA_CORR.gene==gene)]; result["corr_q"]=qrow.q.iloc[0] if len(qrow) else np.nan
    return result


def _phase_alignment_tests():
    counts=MIRNA_PAIRS[MIRNA_PAIRS.both_rhythmic].groupby(['cohort','phase_class']).size().unstack(fill_value=0).reindex(index=list(COHORTS),columns=['anti_phase','in_phase','other'],fill_value=0)
    rows=[]
    for tumor in ['TCGA','CPTAC']:
        a=counts.loc[tumor].to_numpy(int);b=counts.loc['GTEx'].to_numpy(int)
        table=np.vstack([a,b]);table=table[:,table.sum(axis=0)>0]
        chi2,p,dof,expected=stats.chi2_contingency(table,correction=False)
        row=dict(tumor=tumor,normal='GTEx',chi2=chi2,p=p,degrees_freedom=dof,
                 cramers_v=np.sqrt(chi2/table.sum()),minimum_expected=expected.min())
        for group,values in [('tumor',a),('normal',b)]:
            for label,n in zip(['anti_phase','in_phase','other'],values):row[f'{group}_{label}']=int(n)
        rows.append(row)
    out=pd.DataFrame(rows);out['q']=_safeq(out.p)
    return out


def _write_outputs():
    save_table(_phase_alignment_tests(),"mirna_target_phase_alignment_chi_square_tests")
    save_table(MIRNA_PHASE_SUMMARY,"fig4_mirna_phase_density")
    save_table(MIRNA_PAIRS[[c for c in ["cohort","matrix_mirna_id","gene","ensg","mirna_phase","gene_phase","mirna_p","gene_p","mirna_q","gene_q","mirna_rhythmic","gene_rhythmic","both_rhythmic","lag","phase_class","score"] if c in MIRNA_PAIRS]],"fig4_mirna_target_phase_pairs")
    save_table(MIRNA_CORR,"s13_mirna_target_correlations")
    q=pd.DataFrame([dict(cohort=k,n_raw=MIRNA_QC[k]["n_raw"],n_filtered=MIRNA_QC[k]["n_filtered"],n_samples=MIRNA[k].shape[1],n_fit=len(MIRFIT[k]),n_p05=int((MIRFIT[k].p<P_THRESHOLD).sum())) for k in COHORTS])
    save_table(q,"mirna_input_qc")
    mapping_rows=[]; matching_rows=[]
    for cohort in COHORTS:
        target_keys=set(TARGETS.mirna_key)
        for identifier in MIRFIT[cohort].index:
            key=_key(identifier,True)
            mapping_rows.append(dict(cohort=cohort,matrix_mirna_id=identifier,mirna_key=key,has_target_reference=key in target_keys))
        pairs=MIRNA_PAIRS[MIRNA_PAIRS.cohort.eq(cohort)]
        corr=MIRNA_CORR[MIRNA_CORR.cohort.eq(cohort)]
        matching_rows.append(dict(cohort=cohort,tested_mirnas=len(MIRFIT[cohort]),
            tested_mirnas_with_targets=sum(_key(i,True) in target_keys for i in MIRFIT[cohort].index),
            matched_target_pairs=len(pairs),both_rhythmic_pairs=int(pairs.both_rhythmic.sum()),
            selected_correlation_pairs=len(corr),correlation_sample_overlap=len(set(MIRNA[cohort].columns)&set(EXPR[cohort].columns))))
    save_table(pd.DataFrame(mapping_rows),'mirna_identifier_target_mapping')
    save_table(pd.DataFrame(matching_rows),'mirna_target_matching_qc')
    case=CXCL12_CASE.copy(); site=_site_info(case)
    row={k:(v if not isinstance(v,pd.Series) else np.nan) for k,v in case.items()}; row.update({"miRNA_q":getattr(case.get("mirna_fit"),"q",np.nan),"gene_q":getattr(case.get("gene_fit"),"q",np.nan),"apa_q":getattr(case.get("apa_fit"),"q",np.nan),"site_qualifies":site.get("qualifies"),"site_type":site.get("site_type"),"site_detail":site.get("detail")})
    fits=[case.get(k) for k in ['mirna_fit','gene_fit','apa_fit']]
    rhythmic=all(f is not None and bool(f.is_rhythmic) for f in fits)
    lag=float(circular_lag(fits[1].phase,fits[0].phase)) if fits[0] is not None and fits[1] is not None else np.nan
    apa_lag=float(circular_lag(fits[2].phase,fits[1].phase)) if fits[2] is not None and fits[1] is not None else np.nan
    row.update(all_three_rhythmic=rhythmic,target_mirna_lag_h=lag,apa_gene_lag_h=apa_lag,
               anti_phase=bool(abs(abs(lag)-12)<=PHASE_WINDOW_H),
               anticorrelation=bool(case.get('rho',np.nan)<0 and case.get('corr_q',np.nan)<Q_THRESHOLD))
    row['all_selection_criteria']=bool(row['all_three_rhythmic'] and row['anti_phase'] and row['anticorrelation'] and row['site_qualifies'])
    for key,fit in zip(['mirna','gene','apa'],fits):
        if fit is not None:
            for metric in ['phase','amplitude','p','q','n']:row[f'{key}_{metric}']=fit[metric]
    save_table(pd.DataFrame([row]),"fig4_cxcl12_prespecified_case")
    observations=[]
    for kind,values in [('mirna',MIRNA['TCGA'].loc[case['mirna_id']]),('gene',EXPR['TCGA'].loc['CXCL12']),('apa',_series_for_event('TCGA',case['event']))]:
        for sample,value in values.items():observations.append(dict(feature=kind,sample=sample,phase_h=(GENE_PHASE if kind=='gene' else PHASE).get(sample,np.nan),value=value))
    save_table(pd.DataFrame(observations),'fig4_cxcl12_prespecified_observations')
    phase_summary=MIRNA_PAIRS[MIRNA_PAIRS.both_rhythmic].groupby(['cohort','phase_class']).size().rename('n').reset_index()
    phase_summary['fraction']=phase_summary.n/phase_summary.groupby('cohort').n.transform('sum')
    save_table(phase_summary,'mirna_target_phase_alignment_summary')
    print('Prespecified CXCL12 triad passes all selection criteria:',row['all_selection_criteria'],flush=True)
    display(pd.DataFrame([row])[[c for c in row if c.endswith('_p') or c.endswith('_q') or c.endswith('_lag_h') or c in ['all_three_rhythmic','anti_phase','anticorrelation','site_qualifies','all_selection_criteria']]])
    save_table(pd.DataFrame([site]),"fig4_cxcl12_distal_seed_assessment")


def run_mirna_analysis():
    """Load primary miRNA/TargetScan data, recompute fits, pairs and correlations."""
    global MIRNA,MIRFIT,MIRNA_QC,TARGETS,MIRNA_PAIRS,MIRNA_CORR,CXCL12_CASE,MIRNA_PHASE_SUMMARY
    MIRNA={}; MIRFIT={}; MIRNA_QC={}
    for c in COHORTS:
        if c=="GTEx": raw,expr,cross=_gtex_matrix(); kind="GTEx"
        else: raw,expr,cross=_gdc_matrix(DATA_DIR/("miRNA/TCGA" if c=="TCGA" else "miRNA/CPTAC-3_Tumor")); kind="GDC"; _record(DATA_DIR/("miRNA/TCGA" if c=="TCGA" else "miRNA/CPTAC-3_Tumor"))
        common=[x for x in expr.columns if x in PHASE.index]
        raw,expr=raw.loc[:,common],expr.loc[:,common]
        expr=_filter(raw,expr,cross,kind)
        MIRNA[c]=expr; MIRFIT[c]=_fit(expr); MIRFIT[c]['is_rhythmic']=MIRFIT[c].p<P_THRESHOLD; MIRNA_QC[c]={"n_raw":raw.shape[0],"n_filtered":expr.shape[0],"n_samples":expr.shape[1]}
        print(c,'miRNA:',expr.shape,flush=True)
        save_table(MIRFIT[c].rename_axis('mirna_id').reset_index(),f'{c}_mirna_harmonic_fits')
    TARGETS=_targets(); MIRNA_PAIRS=_pair_table(); MIRNA_CORR=_correlations(MIRNA_PAIRS)
    dd=[]
    for c in COHORTS:
        for what,df in [("miRNA",MIRFIT[c]),("gene",GENE_ALL[c])]:
            z=significant(df); theta,density=circular_density(z.phase.to_numpy(),bandwidth=.3)
            dd.append(pd.DataFrame({"theta":theta,"density":density,"cohort":c,"feature":what}))
    MIRNA_PHASE_SUMMARY=pd.concat(dd,ignore_index=True) if dd else pd.DataFrame(); CXCL12_CASE=_cxcl12_case(); _write_outputs()
    print("CXCL12 prespecified eligibility:", {"miRNA_p":getattr(CXCL12_CASE.get("mirna_fit"),"p",np.nan),"gene_q":getattr(CXCL12_CASE.get("gene_fit"),"q",np.nan),"apa_p":getattr(CXCL12_CASE.get("apa_fit"),"p",np.nan),"corr_q":CXCL12_CASE.get("corr_q",np.nan),"site":_site_info(CXCL12_CASE).get("site_type")})
    return dict(mirna=MIRNA,mirfit=MIRFIT,pairs=MIRNA_PAIRS,correlations=MIRNA_CORR,cxcl12=CXCL12_CASE)


def _txtfit(row,stat="p"):
    if row is None: return "unavailable"
    stat=stat.lower()
    if stat not in {"p","q"}:raise ValueError("Use p for raw or q for adjusted values")
    value=row.q if stat=="q" else row.p
    return f"phase {row.phase:.1f} h\n"+f"${stat}$"+_q_label(value)+f"; n={int(row.n)}"

def _curve(ax, values, color, ylabel, title, fit, phase,phase_values=None):
    phase_values=PHASE if phase_values is None else phase_values
    x=pd.Series(values); ids=[s for s in x.index if s in phase_values.index]; xx=np.asarray([phase_values[s] for s in ids],float); yy=pd.to_numeric(x.loc[ids],errors="coerce").to_numpy(); ok=np.isfinite(xx)&np.isfinite(yy)
    ax.scatter(xx[ok],yy[ok],s=10,color=color,alpha=.7); grid=np.linspace(0,24,240)
    if fit is not None and all(k in fit.index for k in ("beta0","beta_cos","beta_sin")):
        ax.plot(grid,float(fit.beta0)+float(fit.beta_cos)*np.cos(_phase_rad(grid))+float(fit.beta_sin)*np.sin(_phase_rad(grid)),color=color,lw=1.2)
    if pd.notna(phase): ax.axvline(phase,color=color,lw=.8,ls="--")
    ax.set(xlim=(0,24),xticks=[0,6,12,18,24],xlabel="CHIRAL phase (h)",ylabel=ylabel,title=title)


def _site_info(case):
    """Canonical TargetScan seed scan in the event's strand-aware 3'UTR.
    `loci` is DaPars chr:start-end; for minus-strand events its lower coordinate
    is a zero-based boundary; start+1 is the 1-based distal PAS.
    Proximal PAS is the median across phase-aligned TCGA sample calls.
    """
    t=TARGETS[(TARGETS.mirna_key=="hsa-mir-135b")&(TARGETS.gene=="CXCL12")]
    seed=t.seed_m8.iloc[0] if len(t) and "seed_m8" in t else None; meta=case.get("meta")
    result=dict(seed=seed,site_type="not assessed",qualifies=False,detail="region/proximal coordinates unavailable")
    if meta is None or seed is None: return result
    fields={str(k).lower():meta[k] for k in meta.index}; event=str(case.get("event", "")); strand=event.split("|")[-1]
    loci=fields.get("loci",None); prox=fields.get("proximal_apa",None)
    if not (isinstance(loci,str) and pd.notna(prox) and strand in "+-"): return result
    hit=re.match(r"([^:]+):(\d+)-(\d+)",loci)
    if not hit: return result
    chrom,start,end=hit.group(1),int(hit.group(2)),int(hit.group(3)); prox=int(float(prox))
    result.update(coordinate_convention="GRCh38, 1-based inclusive seed coordinates; DaPars loci 0-based half-open", proximal_pas=prox, proximal_pas_method="median across matched sample calls", distal_pas=end if strand=="+" else start+1)
    try:
        fasta=record_input(DATA_DIR/"Ref/GRCh38.primary_assembly.genome.fa")
        record_input(Path(str(fasta)+".fai"))
        try:
            import pysam
            fa=pysam.FastaFile(str(fasta)); seq=fa.fetch(str(chrom),int(start),int(end)).upper(); fa.close()
        except ImportError:
            seq=_faidx_fetch(fasta,str(chrom),int(start),int(end))
        if strand=="-": seq=seq.translate(str.maketrans("ACGT","TGCA"))[::-1]
        core7=str(seed).upper().replace("U","T").translate(str.maketrans("ACGT","TGCA"))[::-1]
        core6,m8=core7[1:],core7[0]; sites=[]
        for match in re.finditer("(?="+core6+")",seq):
            off=match.start(); before=seq[off-1] if off else ""; after=seq[off+6] if off+6<len(seq) else ""
            typ="8mer" if before==m8 and after=="A" else "7mer-m8" if before==m8 else "7mer-A1" if after=="A" else "6mer"
            gstart=start+1+off if strand=="+" else end-off-5; gend=gstart+5
            anchor=gstart if strand=="+" else gend; distance=(anchor-prox) if strand=="+" else (prox-anchor)
            sites.append(dict(site_type=typ,start=gstart,end=gend,distance=int(distance),distal=distance>0))
        functional=[s for s in sites if s["distal"] and s["site_type"] in {"8mer","7mer-m8","7mer-A1"}]
        if functional:
            rank={"8mer":3,"7mer-m8":2,"7mer-A1":1}; best=sorted(functional,key=lambda s:rank[s["site_type"]],reverse=True)[0]
            result.update(site_type=best["site_type"],qualifies=True,coordinate=f"{chrom}:{best['start']}-{best['end']}",distance=best["distance"],autr_len=abs((end if strand=="+" else start+1)-prox),detail=f"{chrom}:{best['start']}-{best['end']}; {best['distance']} nt distal to proximal PAS")
        else: result.update(site_type="no functional distal seed",sites=sites,autr_len=abs((end if strand=="+" else start+1)-prox),detail=f"scanned {chrom}:{start}-{end}")
    except Exception as exc:
        result.update(site_type="site scan unavailable",detail=f"{chrom}:{start}-{end}; {type(exc).__name__}")
    return result


def register_mirna_panels():
    """Register Fig4 J-P and S13 A-F; call after run_mirna_analysis()."""
    if not MIRFIT: raise RuntimeError("Call run_mirna_analysis() first")
    def density(canvas, cohort):
        ax=canvas.add_axes([.13,.20,.76,.64],projection="polar")
        for f,col in [("gene","#0072B2"),("miRNA","#D55E00")]:
            d=MIRNA_PHASE_SUMMARY[(MIRNA_PHASE_SUMMARY.cohort==cohort)&(MIRNA_PHASE_SUMMARY.feature==f)]
            ax.plot(d.theta,d.density,color=col,lw=1,label=f)
        for df,col in [(GENE_ALL[cohort],"#0072B2"),(MIRFIT[cohort],"#D55E00")]:
            axis,r2=phase_axis(significant(df).phase)
            if np.isfinite(axis): ax.axvline(axis*np.pi/12,color=col,ls="--",lw=.7); ax.axvline(((axis+12)%24)*np.pi/12,color=col,ls="--",lw=.7)
        style_polar(ax);ax.tick_params(pad=1)
        canvas.text(.5,.99,f"{cohort}: gene / miRNA",ha='center',va='top',fontsize=FONT['title'])
        handles,labels=ax.get_legend_handles_labels()
        canvas.legend(handles,labels,loc='lower center',ncol=2,fontsize=FONT['legend'],borderpad=.2,handlelength=1.3)
    def alignment(canvas):
        ax=canvas.subplots(); z=MIRNA_PAIRS[MIRNA_PAIRS.both_rhythmic].groupby(["cohort","phase_class"]).size().unstack(fill_value=0).reindex(columns=["anti_phase","in_phase"],fill_value=0); z=z.div(MIRNA_PAIRS[MIRNA_PAIRS.both_rhythmic].groupby("cohort").size(),axis=0)
        y=np.arange(len(z)); ax.barh(y-.18,z.anti_phase,.34,label="anti-phase",color="#D55E00"); ax.barh(y+.18,z.in_phase,.34,label="in-phase",color="#0072B2"); ax.set(yticks=y,yticklabels=list(z.index),xlabel="Fraction of rhythmic pairs",xlim=(0,.6)); ax.legend(fontsize=FONT["legend"],loc="upper right",handlelength=.8,handletextpad=.3)
        tests=_phase_alignment_tests()
        labels=['χ² vs GTEx']+[f'{r.tumor} $q${_q_label(r.q)}' for r in tests.itertuples()]
        ax.text(.98,.02,'\n'.join(labels),transform=ax.transAxes,ha='right',va='bottom',fontsize=FONT['annotation'])
        ax.set_title('Target phase alignment');ax.set_ylim(-.5,3.3)
    case=CXCL12_CASE
    def genep(canvas):
        ax=canvas.subplots(); r=case.get("gene_fit"); curve=case.get("gene_display_fit"); _curve(ax,EXPR["TCGA"].loc["CXCL12"],"#0072B2","TMM log2 CPM","CXCL12 expression",curve,getattr(r,"phase",np.nan),phase_values=GENE_PHASE); ax.text(.02,.98,_txtfit(r,"q"),transform=ax.transAxes,va="top",fontsize=FONT["annotation"],bbox=dict(facecolor="white",edgecolor="none",alpha=.8,pad=1))
    def apap(canvas):
        ax=canvas.subplots(); r=case.get("apa_fit"); s=_series_for_event("TCGA",case.get("event")); _curve(ax,s,"#009E73","PDUI","CXCL12 APA",r,getattr(r,"phase",np.nan)); ax.text(.02,.98,_txtfit(r),transform=ax.transAxes,va="top",fontsize=FONT["annotation"],bbox=dict(facecolor="white",edgecolor="none",alpha=.8,pad=1))
    def mirp(canvas):
        ax=canvas.subplots(); r=case.get("mirna_fit"); _curve(ax,MIRNA["TCGA"].loc[case["mirna_id"]],"#D55E00","log2(CPM+1)","hsa-mir-135b",r,getattr(r,"phase",np.nan)); ax.text(.02,.98,_txtfit(r),transform=ax.transAxes,va="top",fontsize=FONT["annotation"],bbox=dict(facecolor="white",edgecolor="none",alpha=.8,pad=1))
    def corr(canvas):
        ax=canvas.subplots(); mi=case.get("mirna_id"); common=[s for s in MIRNA["TCGA"].columns if s in EXPR["TCGA"].columns]; ax.scatter(MIRNA["TCGA"].loc[mi,common],EXPR["TCGA"].loc["CXCL12",common],s=9,c="#555555",alpha=.7); ax.set(xlabel="miR-135b (log2 CPM+1)",ylabel="CXCL12 (TMM log2 CPM)",title="CXCL12–miR-135b"); ax.text(.02,.98,f"rho={case.get('rho',np.nan):.2f}\nBH $q$={case.get('corr_q',np.nan):.2g}\nn={case.get('corr_n',0)}",transform=ax.transAxes,va="top",fontsize=FONT["annotation"],bbox=dict(facecolor="white",edgecolor="none",alpha=.8,pad=1))
        a=MIRNA['TCGA'].loc[mi,common].to_numpy(float);b=EXPR['TCGA'].loc['CXCL12',common].to_numpy(float);ok=np.isfinite(a)&np.isfinite(b)
        slope,intercept=np.polyfit(a[ok],b[ok],1);xx=np.array([a[ok].min(),a[ok].max()]);ax.plot(xx,intercept+slope*xx,color='#D55E00',lw=1)
    def triad(canvas):
        ax=canvas.subplots(); y=[.75,.5,.25]; rows=[("miRNA",case.get("mirna_fit"),"#D55E00"),("gene",case.get("gene_fit"),"#0072B2"),("APA",case.get("apa_fit"),"#009E73")]
        for yy,(lab,r,col) in zip(y,rows):
            ph=getattr(r,"phase",np.nan); ax.scatter(ph,yy,s=35,c=col); ax.text(ph,yy+.08,f"{lab}\n{ph:.1f} h" if pd.notna(ph) else lab,ha="right" if ph>=20 else "left" if ph<=4 else "center",fontsize=FONT["annotation"])
        ax.set(xlim=(0,24),ylim=(0,1),yticks=[],xticks=[0,6,12,18,24],xlabel="CHIRAL acrophase (h)",title="Triad acrophases")
    def volcano(canvas):
        ax=canvas.subplots(); d=MIRFIT["TCGA"].copy(); raw=-np.log10(d.p.clip(lower=np.nextafter(0,1))); yy=raw.clip(upper=15); out=raw>15; ax.scatter(d.loc[~out,"amplitude"],yy[~out],s=8,c=np.where(d.loc[~out,"p"]<P_THRESHOLD,"#D55E00","#999999"),alpha=.7); ax.scatter(d.loc[out,"amplitude"],yy[out],marker="^",s=18,c="#D55E00",edgecolor="k",lw=.3); ax.axhline(-np.log10(P_THRESHOLD),ls="--",c="k",lw=.7); ax.set(xlabel="Amplitude (log2 CPM)",ylabel=r"−log10 $p$ (cap 15)",title="TCGA miRNA rhythmicity")
    for letter,c in zip("BCD",["GTEx","TCGA","CPTAC"]):
        def hist(canvas,c=c):
            ax=canvas.subplots(); d=MIRNA_CORR[MIRNA_CORR.cohort==c]; ax.hist(d.rho,bins=np.linspace(-1,1,31),color=COHORTS[c]["color"]); ax.axvline(0,c="k",lw=.7,ls="--"); ax.set(xlabel="Spearman rho",ylabel="Target pairs",title=COHORTS[c]["label"])
        register_panel("S13",letter,f"{COHORTS[c]['label']} miRNA-target anticorrelation",hist,1,1)
    def corr_case(canvas):
        corr(canvas)
    def seed(canvas):
        ax=canvas.subplots(); si=_site_info(case)
        if si.get("autr_len") and si.get("qualifies"):
            al=float(si["autr_len"]); pos=float(si["distance"]); common=max(.2*al,100); ax.add_patch(Rectangle((-common,.43),common,.14,color="#BBBBBB")); ax.add_patch(Rectangle((0,.43),al,.14,color="#009E73",alpha=.55)); ax.axvline(0,color="k",lw=1); ax.axvline(al,color="#00795C",lw=1); ax.scatter(pos,.50,marker="v",s=55,color="#D55E00"); ax.text(pos,.67,f"{si['site_type']}\n{si['coordinate']}",ha="center",fontsize=FONT["annotation"]); ax.set(xlim=(-common*1.1,al*1.08),ylim=(.15,.95),xlabel="nt from proximal poly(A) site",yticks=[],title="CXCL12 distal aUTR")
            ax.text(-common/2,.32,'Common',ha='center',fontsize=FONT['annotation'])
            ax.text(al/2,.32,'Distal aUTR (long isoform)',ha='center',fontsize=FONT['annotation'])
            ax.text(0,.86,'Proximal PAS',ha='center',fontsize=FONT['annotation'])
            ax.text(al,.86,'Distal PAS',ha='right',fontsize=FONT['annotation'])
        else:
            ax.text(.5,.55,"No functional distal seed site\ncan be drawn from supplied APA metadata",ha="center",va="center",fontsize=FONT["annotation"]); ax.text(.5,.28,f"TargetScan seed: {si['seed']}\n{si['site_type']}\n{si['detail']}",ha="center",va="center",fontsize=FONT["annotation"]); ax.axis("off"); ax.set_title("CXCL12 distal-UTR seed assessment")
    register_panel("4","J","GTEx miRNA versus gene phase density",lambda canvas:density(canvas,"GTEx"),1,1)
    register_panel("4","K","TCGA miRNA versus gene phase density",lambda canvas:density(canvas,"TCGA"),1,1)
    register_panel("4","L","Three-cohort target phase alignment",alignment,1,1)
    register_panel("4","M","Prespecified CXCL12 expression versus phase",genep,1,1)
    register_panel("4","N","Prespecified CXCL12 APA versus phase",apap,1,1)
    register_panel("4","O","Prespecified hsa-mir-135b versus phase",mirp,1,1)
    register_panel("4","P","CXCL12 triad acrophase timeline",triad,1,1)
    register_panel("S13","A","TCGA miRNA rhythmicity amplitude versus p",volcano,1,1)
    register_panel("S13","E","CXCL12 versus hsa-mir-135b correlation",corr_case,1,1)
    register_panel("S13","F","CXCL12 distal UTR seed schematic",seed,2,1)


In [ ]:
# Cell 7: Run miRNA and target analyses and export source tables
run_mirna_analysis()
print("miRNA rhythmicity: nominal P <", P_THRESHOLD)


## S10D source preparation — TMEM183A BAM coverage

The first code cell configures the BAM directory, all four sample paths, GENCODE v49 GTF, and the source-cache option. Original indexed BAMs take precedence. If none are accessible, the notebook may use the saved, checksum-validated BAM-derived depth tables; their samples, phases, PDUI values, harmonic fit, and proximal site must agree with the current analysis. A partial BAM set or a failed BAM validation raises an error.

Select four distinct representatives near the harmonic trough, rising midpoint, peak, and falling midpoint, choosing the smallest absolute PDUI residual among the eight nearest unused samples in circular phase. Selection is recomputed from current TCGA data. The four samples are TCGA-2J-AAB6-01A, TCGA-YB-A89D-01A, TCGA-IB-AAUW-01A, and TCGA-HZ-7926-01A for the supplied inputs.

The ENST00000965092.1 transcript is missing from the supplied t2g lookup. Its identity as TMEM183A / ENSG00000163444.13 is checked directly against GENCODE v49. Coverage is extracted over GRCh38 chr1:[203022500,203024500), using 0-based half-open coordinates for pysam; exported base positions are 1-based. Depth counts use base quality ≥0 with pysam's `all` read filter (unmapped, secondary, QC-failed and duplicate reads excluded). Overlapping mates are counted separately; no smoothing, downsampling or normalization is applied.

The proximal pA marker uses the **current median DaPars call, 203023501**, rather than the former fixed illustrative position 203023481. The stop-codon start (203023038) and distal transcript boundary (203023908) come from GENCODE. All tracks share one depth scale. Source exports include selected-sample/BAM QC, 8,000 observed base-depth rows, transcript features, and source-file checksums. The notebook reads existing indexes and never builds them.

In [ ]:
# Cell 8: Real TMEM183A BAM coverage, transcript annotation, and source-cache validation
TMEM183A_EVENT_TRANSCRIPT = 'ENST00000965092'
TMEM183A_COVERAGE_LOCUS = {'chrom':'chr1', 'start':203022500, 'end':203024500}
TMEM183A_SOURCE_STEMS = ['S10D_TMEM183A_coverage_manifest',
                        'S10D_TMEM183A_per_base_coverage', 'S10D_TMEM183A_gene_model']


def _tmem_existing_index(path):
    path=Path(path)
    return next((p for p in [Path(str(path)+'.bai'),path.with_suffix('.bai'),Path(str(path)+'.csi')]
                 if p.is_file()),None)


def select_tmem_samples():
    """Recompute the four low-residual representatives from current TCGA PDUI."""
    meta=APA_META['TCGA']
    hits=meta.loc[meta.transcript.astype(str).str.split('.').str[0].eq(TMEM183A_EVENT_TRANSCRIPT)]
    if len(hits)!=1:raise ValueError('Expected exactly one ENST00000965092 APA event')
    event=hits.index[0];fit=APA['TCGA'].loc[event]
    # The supplied t2g table omits this newer transcript. Its name is validated
    # against GENCODE below, rather than requiring a missing t2g gene symbol.
    known_gene=str(hits.iloc[0].gene).upper()
    if known_gene not in {'TMEM183A','NA','NAN','NONE',''}:
        raise ValueError(f'Conflicting TMEM183A gene annotation: {known_gene}')
    rows=[]
    for sample,value in PDUI['TCGA'].loc[event].items():
        if sample not in PHASE.index or not np.isfinite(value):continue
        phase=float(PHASE.loc[sample]);fitted=fit.beta0+fit.beta_cos*np.cos(phase*np.pi/12)+fit.beta_sin*np.sin(phase*np.pi/12)
        if np.isfinite(phase):rows.append(dict(sample_id=sample,phase_h=phase,observed_pdui=value,
                                             fitted_pdui=fitted,absolute_residual=abs(value-fitted)))
    candidates=pd.DataFrame(rows);chosen=[];used=set()
    for name,target in [('trough',(fit.phase+12)%24),('rising_midpoint',(fit.phase+18)%24),
                        ('peak',fit.phase),('falling_midpoint',(fit.phase+6)%24)]:
        pool=candidates.loc[~candidates.sample_id.isin(used)].copy()
        delta=abs(pool.phase_h-target)%24;pool['phase_distance_h']=np.minimum(delta,24-delta)
        pool=pool.sort_values(['phase_distance_h','sample_id'],kind='stable').head(8)
        row=pool.sort_values(['absolute_residual','phase_distance_h','sample_id'],kind='stable').iloc[0].to_dict()
        row.update(target=name,target_phase_h=target);chosen.append(row);used.add(row['sample_id'])
    manifest=pd.DataFrame(chosen).sort_values('phase_h').reset_index(drop=True)
    manifest['event']=event;manifest['transcript']=event.split('|')[0];manifest['gene']='TMEM183A'
    manifest['cohort']='TCGA';manifest['fit_phase_h']=fit.phase
    for field in ['beta0','beta_cos','beta_sin']:manifest[field]=fit[field]
    manifest['proximal_pa']=float(hits.iloc[0].proximal_apa)
    return manifest


def _tmem_gene_model(gtf_path):
    """Read GENCODE transcript features; GTF 1-based intervals become half-open."""
    import gzip
    gtf_path=Path(gtf_path)
    if not gtf_path.is_file():raise FileNotFoundError(f'Set TMEM183A_GTF: {gtf_path}')
    opener=gzip.open if str(gtf_path).endswith('.gz') else open
    rows=[];records=[]
    with opener(gtf_path,'rt') as handle:
        for line in handle:
            if line.startswith('#') or TMEM183A_EVENT_TRANSCRIPT+'.' not in line:continue
            fields=line.rstrip().split('\t')
            attrs=dict(re.findall(r'(\w+) "([^"]*)"',fields[8]))
            if attrs.get('transcript_id','').split('.')[0]!=TMEM183A_EVENT_TRANSCRIPT:continue
            if attrs.get('gene_name')!='TMEM183A' or fields[0]!='chr1' or fields[6]!='+':
                raise ValueError('TMEM183A transcript identity, chromosome, or strand mismatch')
            records.append(line)
            if fields[2] in {'exon','CDS','UTR','stop_codon'}:
                rows.append(dict(feature=fields[2],chrom=fields[0],start=int(fields[3])-1,end=int(fields[4]),
                                 strand=fields[6],transcript=attrs['transcript_id'],gene=attrs['gene_name'],
                                 ensg=attrs['gene_id'],gtf_path=str(gtf_path),annotation='GENCODE v49',
                                 coordinate_system='0-based half-open'))
    model=pd.DataFrame(rows)
    if model.empty or not {'exon','CDS','UTR','stop_codon'} <= set(model.feature):
        raise ValueError('GENCODE transcript features are incomplete')
    model['transcript_records_sha256']=hashlib.sha256(''.join(records).encode()).hexdigest()
    model['gtf_bytes']=gtf_path.stat().st_size
    return model


def extract_tmem183a_bam_coverage(manifest):
    """Extract actual aligned A/C/G/T depth from existing indexed genomic BAMs.

    Zero base-quality threshold and pysam read_callback='all' match the legacy
    coverage policy: exclude unmapped, secondary, QC-failed and duplicate reads;
    supplementary alignments and overlapping mates can contribute. No smoothing,
    normalization, or read-depth downsampling is applied.
    """
    import pysam
    locus=TMEM183A_COVERAGE_LOCUS;tables=[];qc=[]
    for row in manifest.itertuples():
        path=Path(BAM_PATHS.get(row.sample_id,TMEM183A_BAM_DIR/f'{row.sample_id}.bam'))
        index=_tmem_existing_index(path)
        if not path.is_file() or index is None:raise FileNotFoundError(f'BAM or existing index missing: {path}')
        pysam.quickcheck(str(path))
        with pysam.AlignmentFile(str(path),'rb',index_filename=str(index)) as bam:
            if bam.header.to_dict().get('HD',{}).get('SO')!='coordinate':
                raise ValueError(f'BAM must be coordinate sorted: {path}')
            if 'chr1' not in bam.references or bam.get_reference_length('chr1')!=248956422:
                raise ValueError(f'Expected GRCh38 chr1 (248956422 bp): {path}')
            if not bam.check_index():raise ValueError(f'Unreadable index: {index}')
            header=bam.header.to_dict();sm=sorted({g.get('SM','') for g in header.get('RG',[])})
            tcga_sm=[v for v in sm if v.startswith('TCGA-')]
            if tcga_sm and not any(v[:12]==row.sample_id[:12] for v in tcga_sm):
                raise ValueError(f'BAM read-group sample mismatch: {row.sample_id}: {sm}')
            depth=np.asarray(bam.count_coverage(locus['chrom'],locus['start'],locus['end'],
                                                quality_threshold=0,read_callback='all'),dtype=np.int64).sum(axis=0)
            header_sha=hashlib.sha256(str(bam.header).encode()).hexdigest()
        if len(depth)!=locus['end']-locus['start'] or depth.max()<=0:
            raise ValueError(f'Coverage absent/incomplete for {row.sample_id}')
        tables.append(pd.DataFrame({'sample_id':row.sample_id,'chrom':locus['chrom'],
                                    'position':np.arange(locus['start']+1,locus['end']+1),
                                    'coverage':depth,'coordinate_system':'1-based',
                                    'coverage_source':'pysam.count_coverage'}))
        qc.append(dict(sample_id=row.sample_id,bam_path=str(path),bam_bytes=path.stat().st_size,
                       bam_mtime_ns=path.stat().st_mtime_ns,index_path=str(index),index_bytes=index.stat().st_size,
                       index_sha256=hashlib.sha256(index.read_bytes()).hexdigest(),bam_header_sha256=header_sha,
                       bam_read_group_samples=';'.join(sm),bam_quickcheck='pass',bam_index_check='pass',
                       bam_sort_order='coordinate',genome='GRCh38',chr1_length=248956422,
                       max_depth=int(depth.max()),covered_bases=int((depth>0).sum()),depth_sum=int(depth.sum()),
                       minimum_base_quality=0,read_callback='all',pysam_version=pysam.__version__,
                       coverage_source='pysam.count_coverage'))
    return pd.concat(tables,ignore_index=True),pd.DataFrame(qc)


def _tmem_validate_sources(manifest,coverage,model,current):
    if len(manifest)!=4 or set(manifest.sample_id)!=set(current.sample_id):
        raise ValueError('BAM source samples do not match the current low-residual selection')
    a=manifest.set_index('sample_id').loc[current.sample_id];b=current.set_index('sample_id')
    for field in ['phase_h','observed_pdui','fitted_pdui','fit_phase_h','proximal_pa']:
        if not np.allclose(a[field].to_numpy(float),b[field].to_numpy(float),rtol=1e-8,atol=1e-10):
            raise ValueError(f'Stale S10D source field: {field}; re-extract BAM sources')
    if not (a.event.to_numpy()==b.event.to_numpy()).all():raise ValueError('S10D event mismatch')
    expected=np.arange(TMEM183A_COVERAGE_LOCUS['start']+1,TMEM183A_COVERAGE_LOCUS['end']+1)
    if set(coverage.sample_id)!=set(current.sample_id):raise ValueError('Unexpected coverage samples')
    for sample,z in coverage.groupby('sample_id'):
        if not np.array_equal(z.sort_values('position').position.to_numpy(),expected):
            raise ValueError(f'Incomplete/duplicated coverage coordinates: {sample}')
        if not np.isfinite(z.coverage).all() or (z.coverage<0).any() or (z.coverage%1!=0).any():
            raise ValueError('Coverage must be nonnegative integer observed depth')
        if z.coverage.max()<=0:raise ValueError('No observed coverage')
        if not z.chrom.eq('chr1').all():raise ValueError('Coverage chromosome mismatch')
    if not model.gene.eq('TMEM183A').all() or not model.transcript.str.split('.').str[0].eq(TMEM183A_EVENT_TRANSCRIPT).all():
        raise ValueError('Transcript annotation mismatch')
    if not model.chrom.eq('chr1').all() or not model.strand.eq('+').all():raise ValueError('Gene locus mismatch')
    if not {'exon','CDS','UTR','stop_codon'} <= set(model.feature):raise ValueError('Incomplete gene model')
    if not manifest.coverage_source.eq('pysam.count_coverage').all():raise ValueError('Missing BAM extraction provenance')


def prepare_tmem183a_coverage():
    current=select_tmem_samples()
    paths=[Path(BAM_PATHS.get(s,TMEM183A_BAM_DIR/f'{s}.bam')) for s in current.sample_id]
    present=[p.is_file() for p in paths]
    source_paths=[TABLE_DIR/f'{stem}.tsv' for stem in TMEM183A_SOURCE_STEMS]
    provenance_path=TABLE_DIR/'S10D_TMEM183A_source_provenance.json'
    if all(present):
        model=_tmem_gene_model(TMEM183A_GTF)
        coverage,qc=extract_tmem183a_bam_coverage(current)
        manifest=current.merge(qc,on='sample_id',validate='one_to_one')
        manifest['gtf_path']=str(TMEM183A_GTF)
        manifest['annotation']='GENCODE v49'
        manifest['stop_codon']=int(model.loc[model.feature.eq('stop_codon'),'start'].min()+1)
        manifest['distal_pa']=int(model.loc[model.feature.eq('exon'),'end'].max())
        _tmem_validate_sources(manifest,coverage,model,current)
        for table,stem in zip([manifest,coverage,model],TMEM183A_SOURCE_STEMS):save_table(table,stem)
        provenance=dict(coverage_source='pysam.count_coverage',genome='GRCh38',
                        interval_0based=TMEM183A_COVERAGE_LOCUS,coordinate_system='coverage 1-based; gene model 0-based half-open',
                        source_sha256={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in source_paths})
        provenance_path.write_text(json.dumps(provenance,indent=2))
        for p in paths:record_input(p);record_input(_tmem_existing_index(p))
        record_input(TMEM183A_GTF);mode='direct indexed BAM extraction'
    elif any(present):
        raise FileNotFoundError('Only some selected BAMs are present; supply all four BAMs and indexes')
    elif TMEM183A_USE_SOURCE_CACHE and all(p.is_file() for p in source_paths+[provenance_path]):
        provenance=json.loads(provenance_path.read_text())
        if provenance['coverage_source']!='pysam.count_coverage':raise ValueError('Invalid coverage provenance')
        for p in source_paths:
            if hashlib.sha256(p.read_bytes()).hexdigest()!=provenance['source_sha256'][p.name]:
                raise ValueError(f'S10D source checksum mismatch: {p.name}')
        manifest,coverage,model=[pd.read_csv(p,sep='\t') for p in source_paths]
        _tmem_validate_sources(manifest,coverage,model,current)
        mode='validated BAM-derived source tables (original BAMs unavailable in this runtime)'
    else:
        raise FileNotFoundError('Configure TMEM183A_BAM_DIR/BAM_PATHS in the first cell; all four indexed BAMs are required')
    for p in source_paths+[provenance_path]:record_input(p)
    print('S10D coverage source:',mode)
    display(manifest[['sample_id','phase_h','observed_pdui','target','max_depth','covered_bases']])
    return manifest,coverage,model,mode


def register_tmem183a_coverage_panel():
    from matplotlib.patches import Rectangle
    from matplotlib.ticker import FuncFormatter,MaxNLocator
    global TMEM183A_COVERAGE_MODE
    manifest,coverage,model,TMEM183A_COVERAGE_MODE=prepare_tmem183a_coverage()
    locus=TMEM183A_COVERAGE_LOCUS
    stop=int(manifest.stop_codon.iloc[0]);distal=int(manifest.distal_pa.iloc[0]);prox=float(manifest.proximal_pa.iloc[0])
    sites=[(stop,'stop codon','#009E73',':'),(prox,'proximal pA','#D55E00','--'),(distal,'distal pA','#555555','--')]
    colors=['#8BB1C0','#7351A3','#783A74','#CFA78D']
    def draw(canvas):
        grid=canvas.add_gridspec(5,1,height_ratios=[1,1,1,1,.60],hspace=.05)
        axes=[];ymax=coverage.coverage.max()*1.18
        for i,row in enumerate(manifest.itertuples()):
            ax=canvas.add_subplot(grid[i,0],sharex=axes[0] if axes else None);axes.append(ax)
            z=coverage.loc[coverage.sample_id.eq(row.sample_id)].sort_values('position')
            ax.fill_between(z.position,z.coverage,step='mid',color=colors[i],alpha=.9,linewidth=0)
            ax.set(xlim=(locus['start'],locus['end']),ylim=(0,ymax),ylabel='Depth')
            ax.yaxis.set_major_locator(MaxNLocator(3));ax.tick_params(labelbottom=False)
            label=f'{row.sample_id}   phase {row.phase_h:.1f} h   PDUI {row.observed_pdui:.2f} (fit {row.fitted_pdui:.2f})'
            ax.text(.005,.96,label,transform=ax.transAxes,va='top',fontsize=FONT['annotation'],
                    bbox=dict(facecolor='white',edgecolor='none',alpha=.85,pad=.5))
            for coordinate,label,color,style in sites:ax.axvline(coordinate,c=color,ls=style,lw=.8)
        axes[0].set_title('TMEM183A (ENST00000965092.1): TCGA RNA-seq coverage')
        gene_ax=canvas.add_subplot(grid[4,0],sharex=axes[0])
        gene_ax.set(ylim=(-.7,1),yticks=[],xlabel='chr1 position (kb; GRCh38)')
        gene_ax.spines[['top','right','left']].set_visible(False)
        gene_ax.hlines(0,max(locus['start'],model.start.min()+1),min(locus['end'],model.end.max()),color='.4',lw=.7)
        for feature,height,color in [('UTR',.25,'#666666'),('CDS',.46,'#0072B2')]:
            for row in model.loc[model.feature.eq(feature)].itertuples():
                left=max(row.start+1,locus['start']);right=min(row.end,locus['end'])
                if right>left:gene_ax.add_patch(Rectangle((left,-height/2),right-left,height,color=color,lw=0))
        for coordinate,label,color,style in sites:
            gene_ax.axvline(coordinate,c=color,ls=style,lw=.8)
            gene_ax.text(coordinate,.55,label,ha='center',fontsize=FONT['annotation'],color=color)
        gene_ax.xaxis.set_major_formatter(FuncFormatter(lambda value,_:f'{value/1000:,.1f}'))
        gene_ax.xaxis.set_major_locator(MaxNLocator(6))
    register_panel('S10','D','tmem183a_rna_seq_coverage',draw,4,3)


In [ ]:
# Cell 9: Register and validate the exact publication panel map
PANELS.clear();EXPORT_RECORDS.clear()
register_apa_panels()
register_mirna_panels()
register_tmem183a_coverage_panel()
expected={'4':list('ABCDEFGHIJKLMNOP'),'S10':list('ABCD'),'S11':list('ABCDE'),'S12':list('ABC'),'S13':list('ABCDEF')}
for figid,letters in expected.items():
    actual=sorted(l for f,l in PANELS if f==figid)
    assert actual==letters,(figid,actual,letters)
print(f'Registered {len(PANELS)} individual panels. S10 includes real BAM coverage in panel D.')


## Figure 4 — APA and miRNA coordination

Layout: A–D (one module each); E/F (one each) + G (two); H (four wide × two high); I–L (one each); M–P (one each). The assembly is 210 × 315 mm. E/F share one machinery legend. B/C annotate raw P values; the underlying tests and rhythmicity cutoffs are unchanged.

In [ ]:
# Cell 10: Render Figure 4 components and assembled figure
export_figure('4')


## Figure S10 — rhythmic APA heatmaps and TMEM183A RNA-seq coverage

Panels A–C show GTEx, TCGA and CPTAC rhythmic APA events. Panel D shows observed per-base depth from four indexed TCGA RNA-seq BAMs, ordered by their inferred CHIRAL phases. All four panels and the complete assembly are exported and displayed below.

The assembled heatmaps are adjacent and share one row-z-score color bar (−2 to +2); individual panel exports retain their own scale for standalone use.

In [ ]:
# Cell 11: Render S10A–D and the complete assembled figure
export_figure('S10')


## Figure S11 — APA amplitude and machinery expression

In [ ]:
# Cell 12: Render S11 components and assembled figure
export_figure('S11')


## Figure S12 — APA–expression phase lags and splicing machinery

Panel C omits nonsignificant annotations; asterisks, when present, indicate BH-adjusted $q < 0.05$ from the existing two-sided Fisher exact tests.

In [ ]:
# Cell 13: Render S12 components and assembled figure
export_figure('S12')


## Figure S13 — Predicted miRNA-target relationships

In [ ]:
# Cell 14: Render S13 components and assembled figure
export_figure('S13')


## Reproducibility record and completion checks

Tables below report the actually generated outputs. All 34 individual panels, including real S10D BAM coverage, and five complete assemblies must have both PNG and PDF files. The input manifest stores relative locations and sizes; checksums pin the phase input and the authoritative gene files. Package versions and output dimensions are recorded. When distributing the public repository, supply these processed inputs or documented access routes, including the processed TargetScan reference and genomic annotations.

In [ ]:
# Cell 15: Record inputs, package versions, and export validation
input_manifest=pd.DataFrame(INPUT_RECORDS.values()).sort_values('relative_path')
save_table(input_manifest,'publication_input_manifest')
exports=pd.DataFrame(EXPORT_RECORDS).drop_duplicates('stem',keep='last')
save_table(exports,'publication_export_manifest')
for field in ('png','pdf'):
    assert all((ROOT/p).exists() for p in exports[field]),f'Missing {field} export'
assert (exports.minimum_font_pt>=7).all()
assert (exports.width_mm/MODULE_MM).isin([1,2,3,4]).all()
assert (~exports.png_has_panel_letters).all() and exports.pdf_has_panel_letters.all()
assert len(exports)==len(PANELS)+5
versions={name:importlib.metadata.version(name) for name in ['numpy','pandas','scipy','statsmodels','matplotlib','IPython']}
versions.update(python=sys.version,platform=platform.platform(),seed=SEED,apa_mirna_p_threshold=P_THRESHOLD,gene_q_threshold=GENE_Q_THRESHOLD,gene_stored_amplitude_min=GENE_AMPLITUDE_MIN,comparison_q_threshold=Q_THRESHOLD)
phase_path=DATA_DIR/'Merged/CHIRAL_dual_penalty_phi_12h.csv'
versions['apa_mirna_phase_sha256']=hashlib.sha256(phase_path.read_bytes()).hexdigest()
versions['gene_phase_sha256']=versions['apa_mirna_phase_sha256']
versions.update(go_background='whole_GOA_annotated_proteome',go_p_threshold=GO_P_THRESHOLD,go_max_terms_per_cohort=GO_MAX_TERMS_PER_COHORT,go_display_score_cap=GO_DISPLAY_SCORE_MAX)
(TABLE_DIR/'session_information.json').write_text(json.dumps(versions,indent=2))
print('Generated',len(exports),'PNG/PDF pairs; every export is displayed above.')
versions['s10d_coverage_mode']=TMEM183A_COVERAGE_MODE
(TABLE_DIR/'session_information.json').write_text(json.dumps(versions,indent=2))
print('S10D:',TMEM183A_COVERAGE_MODE)
display(exports[['figure','panel','width_mm','height_mm','minimum_font_pt','assembly_complete']])
display(pd.Series(versions,name='value'))
